# InstaQuery: fine-tuning a small open LLM for Text-to-SQL

End-to-end project on **one free Colab T4 GPU**:

1. **Data**: Spider (8.6K train questions, 1,034 dev questions on 20 unseen databases) + a synthetic quick-commerce database with 50 business questions
2. **Baselines**: base model zero-shot, fixed few-shot and **RAG** few-shot (dense retrieval of similar examples)
3. **QLoRA fine-tuning**: 4-bit NF4 base + LoRA adapters (Hugging Face transformers, PEFT, bitsandbytes)
4. **Evaluation**: execution accuracy (official Spider comparison logic), difficulty breakdown, error analysis
5. **Quantization**: 4-bit NF4 accuracy vs memory
6. **Serving**: vLLM OpenAI-compatible server, throughput/latency benchmark vs Hugging Face `generate`
7. **Report + demo**: charts, README with results, CV bullets, Gradio app

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.
`RUN_MODE="quick"` is a ~20-minute smoke test; `"full"` is the complete run (~2.5–3 h).
Every stage caches its outputs, so after a disconnect just **Run all** again and it resumes.

## 0. Settings

In [ ]:
#@title Settings
RUN_MODE = "full"          #@param ["quick", "full"]
USE_DRIVE = True           #@param {type:"boolean"}
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  #@param {type:"string"}
DRIVE_DIR = "/content/drive/MyDrive/InstaQuery"  #@param {type:"string"}

## 1. Environment

In [ ]:
import os, subprocess, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
T_START = time.time()
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE - switch the runtime to a T4 GPU before running")
assert gpu, "No GPU found"

In [ ]:
%pip install -q -U transformers peft accelerate bitsandbytes datasets sentence-transformers sqlglot sqlparse nltk gdown huggingface_hub httpx nest_asyncio
!pip uninstall -y -q torchao   # Colab's preinstalled torchao is too old for the new PEFT and is not needed here
import transformers, peft, torch
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = f"{DRIVE_DIR}/outputs_{RUN_MODE}"
else:
    OUTPUT_DIR = f"/content/instaquery_outputs_{RUN_MODE}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Outputs ->", OUTPUT_DIR)

## 2. Project source code
The same files live in the GitHub repo under `src/instaquery/`.

In [ ]:
!mkdir -p /content/instaquery/src/instaquery /content/instaquery/data /content/instaquery/app /content/instaquery/templates /content/instaquery/tests

In [ ]:
%%writefile /content/instaquery/src/instaquery/__init__.py
"""InstaQuery: fine-tuning a small open LLM for Text-to-SQL with QLoRA.

Pipeline: Spider data -> prompt building (schema + question) -> baselines
(zero-shot, few-shot, retrieval-augmented few-shot) -> QLoRA fine-tuning ->
execution-accuracy evaluation -> 4-bit quantization -> vLLM serving + benchmark.
"""

__version__ = "1.0.0"

In [ ]:
%%writefile /content/instaquery/src/instaquery/config.py
"""Central configuration. Every stage reads paths and hyper-parameters from here."""

from __future__ import annotations

import json
import os
from dataclasses import asdict, dataclass, field
from pathlib import Path


@dataclass
class Config:
    # ---- run mode -------------------------------------------------------
    # "quick": tiny subsets to smoke-test the whole pipeline (~20 min on a T4)
    # "full" : complete Spider train/dev (~2-3 h on a T4)
    run_mode: str = "full"

    # ---- paths ----------------------------------------------------------
    work_dir: str = "/content/instaquery_work"      # fast local disk (data, models)
    output_dir: str = "/content/instaquery_outputs"  # results (copied to Drive if mounted)

    # ---- model ----------------------------------------------------------
    base_model: str = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    served_model_name: str = "instaquery"

    # ---- prompt / data --------------------------------------------------
    max_prompt_tokens: int = 1800       # drop sample values from schema above this
    max_seq_len: int = 2048             # prompt + answer during training
    sample_values_per_column: int = 3
    n_shots: int = 3                    # examples for few-shot and RAG variants
    retriever_model: str = "BAAI/bge-small-en-v1.5"

    # ---- QLoRA ----------------------------------------------------------
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    learning_rate: float = 2e-4
    num_epochs: float = 1.0
    per_device_batch_size: int = 4
    grad_accum_steps: int = 4
    warmup_ratio: float = 0.03
    save_steps: int = 100
    seed: int = 42

    # ---- generation / eval ----------------------------------------------
    max_new_tokens: int = 256
    hf_eval_batch_size: int = 16
    sql_timeout_s: float = 15.0

    # ---- serving --------------------------------------------------------
    vllm_version: str = "0.19.0"
    vllm_port: int = 8000
    vllm_gpu_mem: float = 0.85
    vllm_max_model_len: int = 4096
    bench_requests: int = 200
    bench_concurrency: list = field(default_factory=lambda: [1, 8, 32])

    # ---- quick-mode subset sizes ------------------------------------------
    quick_train_examples: int = 400
    quick_dev_examples: int = 100
    quick_max_steps: int = 40

    # ---- optional paid-API reference ----------------------------------------
    api_model: str = "gpt-4o-mini"
    api_base_url: str = "https://api.openai.com/v1"
    api_eval_examples: int = 1034

    # ------------------------------------------------------------------------
    @property
    def quick(self) -> bool:
        return self.run_mode == "quick"

    def path(self, *parts: str) -> Path:
        p = Path(self.work_dir).joinpath(*parts)
        p.parent.mkdir(parents=True, exist_ok=True)
        return p

    def out(self, *parts: str) -> Path:
        p = Path(self.output_dir).joinpath(*parts)
        p.parent.mkdir(parents=True, exist_ok=True)
        return p

    @property
    def spider_dir(self) -> Path:
        return Path(self.work_dir) / "spider"

    @property
    def adapter_dir(self) -> Path:
        return Path(self.output_dir) / "model" / "lora_adapter"

    @property
    def merged_dir(self) -> Path:
        return Path(self.work_dir) / f"merged_model_{self.run_mode}"

    @property
    def domain_db_path(self) -> Path:
        return Path(self.work_dir) / "quickcommerce" / "quickcommerce.sqlite"

    def save(self) -> None:
        self.out("config.json").write_text(json.dumps(asdict(self), indent=2))


def load_config(**overrides) -> Config:
    cfg = Config(**overrides)
    os.makedirs(cfg.work_dir, exist_ok=True)
    os.makedirs(cfg.output_dir, exist_ok=True)
    return cfg

In [ ]:
%%writefile /content/instaquery/src/instaquery/spider_data.py
"""Download and load the Spider 1.0 Text-to-SQL benchmark (Yu et al., EMNLP 2018).

Spider is cross-database: the 20 dev databases never appear in training, so the
dev score measures generalisation to *unseen schemas*.
"""

from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import zipfile
from dataclasses import dataclass
from pathlib import Path

OFFICIAL_GDRIVE_ID = "1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J"  # yale-lily.github.io/spider
HF_MIRROR = ("HAL-9001/spider-databases", "spider_data.zip")
HF_MIRROR_SHA256 = "00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b"


@dataclass
class Example:
    question: str
    query: str
    db_id: str
    split: str = ""
    idx: int = 0

    def to_dict(self) -> dict:
        return {"question": self.question, "query": self.query, "db_id": self.db_id,
                "split": self.split, "idx": self.idx}


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _find_root(base: Path) -> Path | None:
    """Return the folder containing dev.json + tables.json + database/."""
    for dev in base.rglob("dev.json"):
        root = dev.parent
        if (root / "tables.json").exists() and (root / "database").is_dir():
            return root
    return None


def download_spider(dest: Path) -> Path:
    """Download + unzip Spider into ``dest``; returns the data root. Idempotent."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    root = _find_root(dest)
    if root:
        print(f"[spider] already available at {root}")
        return root

    zip_path = dest / "spider_data.zip"
    errors = []
    # 1) official Google Drive release
    if not zip_path.exists():
        try:
            import gdown  # noqa: F401
            print("[spider] downloading official release from Google Drive ...")
            subprocess.run(["gdown", "--fuzzy", "-O", str(zip_path),
                            f"https://drive.google.com/uc?id={OFFICIAL_GDRIVE_ID}"], check=True)
            if not zipfile.is_zipfile(zip_path):
                raise RuntimeError("downloaded file is not a zip (Drive quota page?)")
        except Exception as e:  # pragma: no cover - network dependent
            errors.append(f"gdrive: {e}")
            zip_path.unlink(missing_ok=True)
    # 2) Hugging Face mirror (checksum-verified)
    if not zip_path.exists():
        try:
            from huggingface_hub import hf_hub_download
            print("[spider] downloading Hugging Face mirror ...")
            p = hf_hub_download(repo_id=HF_MIRROR[0], filename=HF_MIRROR[1], repo_type="dataset",
                                local_dir=str(dest))
            if _sha256(Path(p)) != HF_MIRROR_SHA256:
                print("[spider] WARNING: checksum differs from the value recorded in this repo")
            if Path(p) != zip_path:
                shutil.move(p, zip_path)
        except Exception as e:  # pragma: no cover
            errors.append(f"hf: {e}")
    if not zip_path.exists():
        raise RuntimeError("Could not download Spider. Errors: " + " | ".join(errors))

    print("[spider] extracting ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest)
    root = _find_root(dest)
    if root is None:
        raise RuntimeError("Spider zip extracted but dev.json/tables.json/database not found")
    print(f"[spider] ready at {root}")
    return root


def load_split(root: Path, split: str) -> list[Example]:
    """split in {'train_spider', 'train_others', 'dev'}."""
    path = Path(root) / f"{split}.json"
    if not path.exists():
        return []
    rows = json.loads(path.read_text())
    return [Example(question=r["question"].strip(), query=r["query"].strip(), db_id=r["db_id"],
                    split=split, idx=i) for i, r in enumerate(rows)]


def db_path(root: Path, db_id: str) -> Path:
    return Path(root) / "database" / db_id / f"{db_id}.sqlite"


def load_train(root: Path) -> list[Example]:
    return load_split(root, "train_spider") + load_split(root, "train_others")


def load_dev(root: Path) -> list[Example]:
    return load_split(root, "dev")

In [ ]:
%%writefile /content/instaquery/src/instaquery/schema.py
"""Turn a SQLite database into a compact, LLM-friendly schema description.

Format (one CREATE TABLE per table, with a few example values for text columns
so the model can match literals such as 'France' vs 'france'):

    CREATE TABLE singer (
      singer_id INTEGER PRIMARY KEY,
      name TEXT, -- e.g. 'Joe Sharp', 'Timbaland'
      country TEXT, -- e.g. 'Netherlands', 'United States'
      FOREIGN KEY (x) REFERENCES other(y)
    );
"""

from __future__ import annotations

import re
import sqlite3
from functools import lru_cache
from pathlib import Path

_SAFE_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
_SQL_KEYWORDS = {"order", "group", "select", "from", "where", "table", "index", "limit", "join",
                 "union", "values", "by", "as", "on", "in", "is", "not", "and", "or", "having",
                 "default", "check", "primary", "key", "foreign", "references", "unique", "match"}
_TEXTY = ("char", "text", "clob", "varchar", "string")


def quote_ident(name: str) -> str:
    if _SAFE_IDENT.match(name) and name.lower() not in _SQL_KEYWORDS:
        return name
    return '"' + name.replace('"', '""') + '"'


def connect_readonly(db_file: str | Path, timeout: float = 30.0) -> sqlite3.Connection:
    uri = f"file:{Path(db_file).resolve()}?mode=ro"
    conn = sqlite3.connect(uri, uri=True, timeout=timeout, check_same_thread=False)
    conn.text_factory = lambda b: b.decode(errors="ignore")
    return conn


def _sample_values(cur, table: str, col: str, k: int) -> list[str]:
    try:
        cur.execute(
            f"SELECT DISTINCT {quote_ident(col)} FROM {quote_ident(table)} "
            f"WHERE {quote_ident(col)} IS NOT NULL AND TRIM(CAST({quote_ident(col)} AS TEXT)) != '' "
            f"LIMIT {k}")
        vals = []
        for (v,) in cur.fetchall():
            s = str(v).replace("\n", " ").strip()
            if len(s) > 40:
                s = s[:37] + "..."
            vals.append("'" + s.replace("'", "''") + "'")
        return vals
    except sqlite3.Error:
        return []


@lru_cache(maxsize=512)
def serialize_schema(db_file: str, sample_values: int = 3) -> str:
    conn = connect_readonly(db_file)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY rowid")
    tables = [r[0] for r in cur.fetchall()]
    blocks = []
    for t in tables:
        cur.execute(f"PRAGMA table_info({quote_ident(t)})")
        cols = cur.fetchall()  # cid, name, type, notnull, dflt, pk
        pk_cols = [c[1] for c in cols if c[5]]
        items: list[tuple[str, str]] = []  # (definition, trailing comment)
        for _, name, ctype, _nn, _d, pk in cols:
            ctype = (ctype or "").strip() or "TEXT"
            definition = f"{quote_ident(name)} {ctype}"
            if pk and len(pk_cols) == 1:
                definition += " PRIMARY KEY"
            comment = ""
            if sample_values and any(x in ctype.lower() for x in _TEXTY):
                vals = _sample_values(cur, t, name, sample_values)
                if vals:
                    comment = " -- e.g. " + ", ".join(vals)
            items.append((definition, comment))
        if len(pk_cols) > 1:
            items.append((f"PRIMARY KEY ({', '.join(quote_ident(c) for c in pk_cols)})", ""))
        try:
            cur.execute(f"PRAGMA foreign_key_list({quote_ident(t)})")
            for fk in cur.fetchall():  # id, seq, table, from, to, ...
                ref_t, frm, to = fk[2], fk[3], fk[4]
                to_txt = f"({quote_ident(to)})" if to else ""
                items.append((f"FOREIGN KEY ({quote_ident(frm)}) REFERENCES {quote_ident(ref_t)}{to_txt}", ""))
        except sqlite3.Error:
            pass
        lines = [f"  {d}{',' if i < len(items) - 1 else ''}{c}" for i, (d, c) in enumerate(items)]
        blocks.append(f"CREATE TABLE {quote_ident(t)} (\n" + "\n".join(lines) + "\n);")
    conn.close()
    return "\n\n".join(blocks)


def table_names(db_file: str | Path) -> list[str]:
    conn = connect_readonly(db_file)
    rows = conn.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'").fetchall()
    conn.close()
    return [r[0] for r in rows]

In [ ]:
%%writefile /content/instaquery/src/instaquery/prompts.py
"""Prompt construction and SQL post-processing (shared by training, eval and serving)."""

from __future__ import annotations

import re

SYSTEM_PROMPT = (
    "You are an expert data analyst who writes correct SQLite queries. "
    "Given a database schema and a question, reply with exactly one SQLite query that answers it. "
    "Return only the SQL, with no explanation and no markdown."
)


def format_examples(examples: list[dict] | None) -> str:
    if not examples:
        return ""
    parts = ["Here are example questions with their correct SQL from other databases "
             "(use them only as a guide to SQL patterns):"]
    for ex in examples:
        parts.append(f"Q: {ex['question']}\nSQL: {normalize_sql(ex['query'])}")
    return "\n\n".join(parts) + "\n\n"


def build_user_prompt(schema: str, question: str, examples: list[dict] | None = None) -> str:
    return (f"Database schema:\n{schema}\n\n"
            f"{format_examples(examples)}"
            f"Question: {question}\n\n"
            "Write one SQLite query that answers the question. Return only the SQL.")


def build_messages(schema: str, question: str, examples: list[dict] | None = None,
                   answer: str | None = None) -> list[dict]:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(schema, question, examples)}]
    if answer is not None:
        msgs.append({"role": "assistant", "content": normalize_sql(answer)})
    return msgs


# --------------------------------------------------------------------------
_QUOTED = re.compile(r"('(?:[^']|'')*'|\"(?:[^\"]|\"\")*\")")


def normalize_sql(sql: str) -> str:
    """Collapse whitespace outside string literals and drop the trailing semicolon."""
    sql = sql.strip().rstrip(";").strip()
    out = []
    for i, part in enumerate(_QUOTED.split(sql)):
        out.append(part if i % 2 == 1 else re.sub(r"\s+", " ", part))
    return "".join(out).strip()


_FENCE = re.compile(r"```(?:sql|sqlite)?\s*(.*?)```", re.S | re.I)
_START = re.compile(r"\b(WITH|SELECT)\b", re.I)


def _cut_at_semicolon(sql: str) -> str:
    in_s = in_d = False
    for i, ch in enumerate(sql):
        if ch == "'" and not in_d:
            in_s = not in_s
        elif ch == '"' and not in_s:
            in_d = not in_d
        elif ch == ";" and not in_s and not in_d:
            return sql[:i]
    return sql


def extract_sql(text: str) -> str:
    """Pull the first SQL statement out of a model response."""
    if text is None:
        return ""
    t = text.strip()
    m = _FENCE.search(t)
    if m:
        t = m.group(1).strip()
    elif t.startswith("```"):
        t = t.strip("`").strip()
        if t.lower().startswith(("sql", "sqlite")):
            t = t.split("\n", 1)[1] if "\n" in t else ""
    t = re.sub(r"^\s*(SQL|SQLite|Answer|Query)\s*:\s*", "", t, flags=re.I)
    m = _START.search(t)
    if m and m.start() > 0 and not t[: m.start()].strip().lower().startswith(("with", "select")):
        t = t[m.start():]
    t = _cut_at_semicolon(t)
    # stop at an obvious explanation paragraph
    t = re.split(r"\n\s*\n(?=[A-Z][a-z])", t)[0]
    return normalize_sql(t)

In [ ]:
%%writefile /content/instaquery/src/instaquery/retrieval.py
"""Retrieval of similar (question, SQL) training examples for retrieval-augmented few-shot prompting.

Default: dense embeddings (BAAI/bge-small-en-v1.5 via sentence-transformers).
Fallback: TF-IDF cosine similarity (no model download needed).
"""

from __future__ import annotations

import numpy as np


class ExampleRetriever:
    def __init__(self, pool: list[dict], model_name: str | None = "BAAI/bge-small-en-v1.5",
                 device: str | None = None):
        self.pool = pool
        self.questions = [p["question"] for p in pool]
        self.method = "tfidf"
        self._st = None
        if model_name:
            try:
                from sentence_transformers import SentenceTransformer
                self._st = SentenceTransformer(model_name, device=device)
                self._emb = self._encode(self.questions)
                self.method = f"dense:{model_name}"
            except Exception as e:  # pragma: no cover - depends on network/model
                print(f"[retriever] dense model unavailable ({e}); using TF-IDF")
                self._st = None
        if self._st is None:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self._tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
            self._mat = self._tfidf.fit_transform(self.questions)

    def _encode(self, texts: list[str]) -> np.ndarray:
        return np.asarray(self._st.encode(texts, batch_size=128, normalize_embeddings=True,
                                          show_progress_bar=False), dtype=np.float32)

    def retrieve_many(self, questions: list[str], k: int = 3, exclude_db: list[str] | None = None) -> list[list[dict]]:
        if self._st is not None:
            q = self._encode(questions)
            sims = q @ self._emb.T
        else:
            sims = (self._tfidf.transform(questions) @ self._mat.T).toarray()
        results = []
        for i in range(len(questions)):
            order = np.argsort(-sims[i])
            picked, seen_sql = [], set()
            for j in order:
                cand = self.pool[int(j)]
                if exclude_db and cand["db_id"] == exclude_db[i]:
                    continue  # never leak the same database
                if cand["query"] in seen_sql:
                    continue
                picked.append(cand)
                seen_sql.add(cand["query"])
                if len(picked) == k:
                    break
            results.append(picked)
        return results


def pick_fixed_shots(train: list[dict], k: int = 3) -> list[dict]:
    """Deterministically pick k diverse examples from the *training* set:
    a filter+aggregate, a GROUP BY/HAVING, and a JOIN with ORDER BY/LIMIT."""
    import re
    patterns = [r"^SELECT count\(\*\) FROM \w+ WHERE",
                r"GROUP BY .* HAVING",
                r"JOIN .* ORDER BY .* LIMIT 1$"]
    shots = []
    for pat in patterns:
        for ex in train:
            q = " ".join(ex["query"].split())
            if re.search(pat, q, re.I) and len(q) < 160 and ex not in shots:
                shots.append(ex)
                break
    for ex in train:  # top up if a pattern was not found
        if len(shots) >= k:
            break
        if ex not in shots:
            shots.append(ex)
    return shots[:k]

In [ ]:
%%writefile /content/instaquery/src/instaquery/sql_eval.py
"""Execution accuracy (EX) for Text-to-SQL.

A prediction is correct if, executed on the database, it returns the same
result as the gold query. The comparison (`result_eq`) follows the official
Spider test-suite evaluator (taoyds/test-suite-sql-eval, Apache-2.0):
bag semantics, row order only matters when the gold query has ORDER BY, and
columns may be permuted. DISTINCT is removed from both queries before
execution, as the official script does by default.

Unlike the official script, queries here have a hard timeout (via SQLite's
progress handler) so a runaway cartesian join cannot hang the evaluation.
"""

from __future__ import annotations

import random
import re
import sqlite3
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from itertools import product
from pathlib import Path

from .schema import connect_readonly

MAX_ROWS = 100_000


# ---------------------------------------------------------------- execution
def execute_sql(db_file: str | Path, sql: str, timeout_s: float = 15.0):
    """Returns ("ok", rows) or ("error", message)."""
    if not sql or not sql.strip():
        return "error", "empty query"
    conn = connect_readonly(db_file)
    deadline = time.monotonic() + timeout_s
    conn.set_progress_handler(lambda: 1 if time.monotonic() > deadline else 0, 10_000)
    try:
        cur = conn.execute(sql)
        rows = cur.fetchmany(MAX_ROWS + 1)
        if len(rows) > MAX_ROWS:
            return "error", f"result too large (> {MAX_ROWS} rows)"
        return "ok", rows
    except sqlite3.OperationalError as e:
        msg = str(e)
        return "error", ("timeout" if "interrupted" in msg else msg)
    except Exception as e:  # sqlite3.Warning for multiple statements, etc.
        return "error", f"{type(e).__name__}: {e}"
    finally:
        conn.close()


# ---------------------------------------------------- official comparison logic
def _unorder_row(row):
    return tuple(sorted(row, key=lambda x: str(x) + str(type(x))))


def _quick_rej(r1, r2, order_matters):
    s1 = [_unorder_row(r) for r in r1]
    s2 = [_unorder_row(r) for r in r2]
    return s1 == s2 if order_matters else set(s1) == set(s2)


def _multiset_eq(l1, l2):
    if len(l1) != len(l2):
        return False
    d = defaultdict(int)
    for e in l1:
        d[e] += 1
    for e in l2:
        d[e] -= 1
        if d[e] < 0:
            return False
    return True


def _constraint_permutations(tab1_sets_by_columns, result2, rng):
    num_cols = len(result2[0])
    perm_constraints = [set(range(num_cols)) for _ in range(num_cols)]
    if num_cols <= 3:
        return product(*perm_constraints)
    for _ in range(20):
        row = rng.choice(result2)
        for c1 in range(num_cols):
            for c2 in set(perm_constraints[c1]):
                if row[c2] not in tab1_sets_by_columns[c1]:
                    perm_constraints[c1].remove(c2)
    return product(*perm_constraints)


def result_eq(result1, result2, order_matters: bool) -> bool:
    if len(result1) == 0 and len(result2) == 0:
        return True
    if len(result1) != len(result2):
        return False
    num_cols = len(result1[0])
    if len(result2[0]) != num_cols:
        return False
    if not _quick_rej(result1, result2, order_matters):
        return False
    tab1_sets = [{row[i] for row in result1} for i in range(num_cols)]
    rng = random.Random(0)
    for perm in _constraint_permutations(tab1_sets, result2, rng):
        if len(perm) != len(set(perm)):
            continue
        r2p = result2 if num_cols == 1 else [tuple(e[i] for i in perm) for e in result2]
        if order_matters:
            if result1 == r2p:
                return True
        elif set(result1) == set(r2p) and _multiset_eq(result1, r2p):
            return True
    return False


_DISTINCT = re.compile(r"\bdistinct\b", re.I)


def remove_distinct(sql: str) -> str:
    try:
        import sqlparse
        toks = [t.value for t in sqlparse.parse(sql)[0].flatten()]
        return "".join(t for t in toks if t.lower() != "distinct")
    except Exception:
        return _DISTINCT.sub("", sql)


def _postprocess(q: str) -> str:
    return q.replace("> =", ">=").replace("< =", "<=").replace("! =", "!=")


# ------------------------------------------------------------------ scoring
@dataclass
class ExecResult:
    correct: bool
    pred_executable: bool
    error: str = ""
    gold_rows: int = 0
    pred_rows: int = 0


def exec_match(db_file: str | Path, pred_sql: str, gold_sql: str, timeout_s: float = 15.0,
               keep_distinct: bool = False) -> ExecResult:
    p, g = _postprocess(pred_sql), _postprocess(gold_sql)
    if not keep_distinct:
        p, g = remove_distinct(p), remove_distinct(g)
    order_matters = "order by" in g.lower()
    g_flag, g_res = execute_sql(db_file, g, timeout_s)
    if g_flag != "ok":
        # gold must execute; treat as evaluation-data problem
        return ExecResult(False, False, f"GOLD FAILED: {g_res}")
    p_flag, p_res = execute_sql(db_file, p, timeout_s)
    if p_flag != "ok":
        return ExecResult(False, False, str(p_res), len(g_res), 0)
    ok = result_eq(g_res, p_res, order_matters)
    return ExecResult(ok, True, "" if ok else "wrong result", len(g_res), len(p_res))


# -------------------------------------------------------- error categories
def categorize_error(r: ExecResult) -> str:
    if r.correct:
        return "correct"
    e = (r.error or "").lower()
    if e.startswith("gold failed"):
        return "gold query failed"
    if not r.pred_executable:
        if "no such column" in e:
            return "unknown column"
        if "no such table" in e:
            return "unknown table"
        if "syntax error" in e or "incomplete input" in e or "unrecognized token" in e:
            return "syntax error"
        if "timeout" in e:
            return "timeout"
        if "ambiguous" in e:
            return "ambiguous column"
        if "empty" in e:
            return "empty output"
        return "other execution error"
    if r.pred_rows != r.gold_rows:
        return "wrong result (row count)"
    return "wrong result (values)"


@dataclass
class Scored:
    records: list = field(default_factory=list)

    @property
    def ex(self) -> float:
        return sum(r["correct"] for r in self.records) / max(1, len(self.records))

    @property
    def exec_rate(self) -> float:
        return sum(r["pred_executable"] for r in self.records) / max(1, len(self.records))

    def by(self, key: str) -> dict:
        groups = defaultdict(list)
        for r in self.records:
            groups[r.get(key, "n/a")].append(r["correct"])
        return {k: (sum(v) / len(v), len(v)) for k, v in groups.items()}

    def error_counts(self) -> dict:
        return dict(Counter(r["category"] for r in self.records))


def score(examples: list[dict], preds: list[str], db_lookup, timeout_s: float = 15.0,
          extra: list[dict] | None = None, progress: bool = True, keep_distinct: bool = False) -> Scored:
    """examples: dicts with question/query/db_id (+ optional hardness). db_lookup(db_id)->path."""
    assert len(examples) == len(preds)
    it = range(len(examples))
    if progress:
        try:
            from tqdm.auto import tqdm
            it = tqdm(it, desc="executing", leave=False)
        except ImportError:
            pass
    out = Scored()
    for i in it:
        ex, pred = examples[i], preds[i]
        r = exec_match(db_lookup(ex["db_id"]), pred, ex["query"], timeout_s, keep_distinct)
        rec = {"idx": i, "db_id": ex["db_id"], "question": ex["question"], "gold": ex["query"],
               "pred": pred, "correct": bool(r.correct), "pred_executable": bool(r.pred_executable),
               "error": r.error, "category": categorize_error(r),
               "hardness": ex.get("hardness", ex.get("difficulty", "n/a"))}
        if extra:
            rec.update(extra[i])
        out.records.append(rec)
    return out

In [ ]:
%%writefile /content/instaquery/src/instaquery/official_eval.py
"""Thin wrapper around the official Spider evaluator (taoyds/test-suite-sql-eval).

Used for two things:
  * Spider's official difficulty label (easy / medium / hard / extra) per question
  * the official Exact-Set-Match (EM) score, reported next to execution accuracy
"""

from __future__ import annotations

import re
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/taoyds/test-suite-sql-eval"


def ensure_repo(dest: Path) -> Path:
    dest = Path(dest)
    if not (dest / "evaluation.py").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, str(dest)], check=True)
    # The official parser calls nltk.word_tokenize, which normally needs the 'punkt' sentence model.
    # A SQL query is a single "sentence", so we patch it to word_tokenize(s, preserve_line=True),
    # which gives the same tokens without downloading punkt (blocked behind some proxies).
    patch = dest / "_nltk_patch.py"
    patch.write_text(
        "import nltk, process_sql\n"
        "process_sql.word_tokenize = lambda s: nltk.word_tokenize(s, preserve_line=True)\n")
    return dest


def _patch_tokenizer():
    import nltk
    import process_sql  # type: ignore
    process_sql.word_tokenize = lambda s: nltk.word_tokenize(s, preserve_line=True)


def add_hardness(examples: list[dict], repo: Path, db_lookup) -> list[dict]:
    """Adds ex['hardness'] using the official parser. Falls back to 'n/a'."""
    repo = str(Path(repo).resolve())
    if repo not in sys.path:
        sys.path.insert(0, repo)
    try:
        _patch_tokenizer()
        from process_sql import Schema, get_schema, get_sql  # type: ignore
        from evaluation import Evaluator  # type: ignore
    except Exception as e:
        print("[official-eval] could not import official evaluator:", e)
        for ex in examples:
            ex.setdefault("hardness", "n/a")
        return examples
    ev, cache = Evaluator(), {}
    for ex in examples:
        try:
            db = str(db_lookup(ex["db_id"]))
            if db not in cache:
                cache[db] = Schema(get_schema(db))
            ex["hardness"] = ev.eval_hardness(get_sql(cache[db], ex["query"]))
        except Exception:
            ex["hardness"] = "n/a"
    return examples


def exact_match(gold: list[dict], preds: list[str], repo: Path, db_dir: Path, tables_json: Path,
                work: Path) -> dict | None:
    """Runs the official script with --etype match; returns {'easy':..,'all':..} or None."""
    work = Path(work)
    work.mkdir(parents=True, exist_ok=True)
    gold_f, pred_f = work / "gold.txt", work / "pred.txt"
    gold_f.write_text("\n".join(f"{g['query']}\t{g['db_id']}" for g in gold) + "\n")
    pred_f.write_text("\n".join((p.replace("\n", " ").replace("\t", " ") or "SELECT 1") for p in preds) + "\n")
    try:
        argv = ["evaluation.py", "--gold", str(gold_f.resolve()), "--pred", str(pred_f.resolve()),
                "--etype", "match", "--db", str(Path(db_dir).resolve()), "--table", str(Path(tables_json).resolve())]
        runner = ("import sys, runpy, _nltk_patch; sys.argv = " + repr(argv) +
                  "; runpy.run_path('evaluation.py', run_name='__main__')")
        out = subprocess.run([sys.executable, "-c", runner], cwd=str(repo), capture_output=True, text=True,
                             timeout=1800)
        text = out.stdout
        (work / "official_match_output.txt").write_text(text + "\n" + out.stderr[-5000:])
        m = re.search(r"^\s*exact match\s+([\d.\s]+)$", text, re.M)
        if not m:
            return None
        vals = [float(x) for x in m.group(1).split()]
        keys = ["easy", "medium", "hard", "extra", "all"]
        return dict(zip(keys, vals[:5]))
    except Exception as e:  # pragma: no cover
        print("[official-eval] exact match failed:", e)
        return None

In [ ]:
%%writefile /content/instaquery/src/instaquery/generation.py
"""Two interchangeable generation back-ends with the same interface:

* HFGenerator      - Hugging Face transformers, batched greedy decoding on the GPU
* OpenAIGenerator  - any OpenAI-compatible endpoint (vLLM server, paid API)

Both take chat `messages` lists and return raw text plus timing/token stats.
"""

from __future__ import annotations

import asyncio
import time
from dataclasses import dataclass


@dataclass
class GenOutput:
    text: str
    latency_s: float
    prompt_tokens: int = 0
    completion_tokens: int = 0


# ======================================================================= HF
def _dtype_kwargs(dtype):
    import transformers
    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
    return {"dtype": dtype} if (major, minor) >= (4, 56) else {"torch_dtype": dtype}


def load_hf_model(model_id: str, quant_4bit: bool = False, device_map="auto"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kwargs = dict(device_map=device_map)
    if quant_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16)
    kwargs.update(_dtype_kwargs(torch.float16))
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    return model, tok


class HFGenerator:
    def __init__(self, model, tokenizer, batch_size: int = 16, max_new_tokens: int = 256):
        self.model, self.tok = model, tokenizer
        self.batch_size, self.max_new_tokens = batch_size, max_new_tokens

    def render(self, messages: list[dict]) -> str:
        return self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def generate(self, batch_messages: list[list[dict]], desc: str = "generating") -> list[GenOutput]:
        import torch
        from tqdm.auto import tqdm

        prompts = [self.render(m) for m in batch_messages]
        order = sorted(range(len(prompts)), key=lambda i: -len(prompts[i]))
        outs: list[GenOutput | None] = [None] * len(prompts)
        self.tok.padding_side = "left"
        for s in tqdm(range(0, len(order), self.batch_size), desc=desc, leave=False):
            idx = order[s:s + self.batch_size]
            enc = self.tok([prompts[i] for i in idx], return_tensors="pt", padding=True,
                           add_special_tokens=False).to(self.model.device)
            t0 = time.perf_counter()
            with torch.inference_mode():
                gen = self.model.generate(**enc, max_new_tokens=self.max_new_tokens, do_sample=False,
                                          repetition_penalty=1.0, pad_token_id=self.tok.pad_token_id)
            dt = (time.perf_counter() - t0) / len(idx)
            new = gen[:, enc["input_ids"].shape[1]:]
            for row, i in enumerate(idx):
                ids = new[row]
                n_out = int((ids != self.tok.pad_token_id).sum())
                outs[i] = GenOutput(self.tok.decode(ids, skip_special_tokens=True), dt,
                                    int(enc["attention_mask"][row].sum()), n_out)
        return outs  # type: ignore

    def generate_one_timed(self, messages: list[dict]) -> GenOutput:
        """Unbatched request (used to compare single-request latency with vLLM)."""
        import torch
        enc = self.tok([self.render(messages)], return_tensors="pt", add_special_tokens=False).to(self.model.device)
        t0 = time.perf_counter()
        with torch.inference_mode():
            gen = self.model.generate(**enc, max_new_tokens=self.max_new_tokens, do_sample=False,
                                      repetition_penalty=1.0, pad_token_id=self.tok.pad_token_id)
        dt = time.perf_counter() - t0
        new = gen[0, enc["input_ids"].shape[1]:]
        return GenOutput(self.tok.decode(new, skip_special_tokens=True), dt,
                         int(enc["input_ids"].shape[1]), int(new.shape[0]))


# =================================================================== OpenAI
class OpenAIGenerator:
    def __init__(self, base_url: str, model: str, api_key: str = "EMPTY", max_new_tokens: int = 256,
                 concurrency: int = 32, timeout: float = 300.0, extra_body: dict | None = None):
        self.extra_body = extra_body or {}
        self.base_url = base_url.rstrip("/")
        self.model, self.api_key = model, api_key
        self.max_new_tokens, self.concurrency, self.timeout = max_new_tokens, concurrency, timeout

    async def _one(self, client, sem, messages):
        payload = {"model": self.model, "messages": messages, "temperature": 0.0,
                   "max_tokens": self.max_new_tokens, **self.extra_body}
        async with sem:
            for attempt in range(4):
                t0 = time.perf_counter()
                try:
                    r = await client.post(f"{self.base_url}/chat/completions", json=payload)
                    if r.status_code == 429 or r.status_code >= 500:
                        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
                    r.raise_for_status()
                    data = r.json()
                    dt = time.perf_counter() - t0
                    usage = data.get("usage") or {}
                    return GenOutput(data["choices"][0]["message"]["content"] or "", dt,
                                     usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0))
                except Exception as e:
                    if attempt == 3:
                        return GenOutput(f"-- request failed: {e}", time.perf_counter() - t0)
                    await asyncio.sleep(2 ** attempt)

    async def _run(self, batch_messages, concurrency):
        import httpx
        sem = asyncio.Semaphore(concurrency)
        headers = {"Authorization": f"Bearer {self.api_key}"}
        async with httpx.AsyncClient(timeout=self.timeout, headers=headers) as client:
            tasks = [self._one(client, sem, m) for m in batch_messages]
            try:
                from tqdm.asyncio import tqdm_asyncio
                return await tqdm_asyncio.gather(*tasks, leave=False)
            except ImportError:
                return await asyncio.gather(*tasks)

    def generate(self, batch_messages: list[list[dict]], desc: str = "", concurrency: int | None = None):
        conc = concurrency or self.concurrency
        try:
            asyncio.get_running_loop()
            in_loop = True   # Jupyter/Colab already runs an event loop in this thread
        except RuntimeError:
            in_loop = False
        if in_loop:
            from concurrent.futures import ThreadPoolExecutor
            with ThreadPoolExecutor(1) as ex:
                return ex.submit(asyncio.run, self._run(batch_messages, conc)).result()
        return asyncio.run(self._run(batch_messages, conc))

    def timed_batch(self, batch_messages, concurrency: int) -> dict:
        t0 = time.perf_counter()
        outs = self.generate(batch_messages, concurrency=concurrency)
        wall = time.perf_counter() - t0
        lat = sorted(o.latency_s for o in outs)
        toks = sum(o.completion_tokens for o in outs)
        return {"concurrency": concurrency, "requests": len(outs), "wall_s": wall,
                "req_per_s": len(outs) / wall, "output_tok_per_s": toks / wall,
                "p50_latency_s": lat[len(lat) // 2], "p95_latency_s": lat[int(len(lat) * 0.95) - 1],
                "failed": sum(o.text.startswith("-- request failed") for o in outs)}

In [ ]:
%%writefile /content/instaquery/src/instaquery/train.py
"""QLoRA supervised fine-tuning (Hugging Face transformers + PEFT + bitsandbytes).

* Base model loaded in 4-bit NF4 with double quantisation (QLoRA, Dettmers et al. 2023)
* LoRA adapters (r=16, alpha=32) on all attention + MLP projections
* Loss only on the SQL answer tokens (prompt tokens are masked with -100)
* Checkpoints every `save_steps`, so a Colab disconnect can resume
"""

from __future__ import annotations

import json
import math
import time
from pathlib import Path

from .prompts import build_messages


def build_features(examples: list[dict], tokenizer, schema_for, max_seq_len: int, max_prompt_tokens: int):
    """Tokenise chat-formatted examples; mask the prompt so loss is on the SQL only."""
    feats, skipped = [], 0
    for ex in examples:
        schema = schema_for(ex["db_id"], True)
        msgs = build_messages(schema, ex["question"], answer=ex["query"])
        prompt_txt = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
        p_ids = tokenizer(prompt_txt, add_special_tokens=False)["input_ids"]
        if len(p_ids) > max_prompt_tokens:  # retry without sample values
            schema = schema_for(ex["db_id"], False)
            msgs = build_messages(schema, ex["question"], answer=ex["query"])
            prompt_txt = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
            p_ids = tokenizer(prompt_txt, add_special_tokens=False)["input_ids"]
        full_txt = tokenizer.apply_chat_template(msgs, tokenize=False)
        if not full_txt.startswith(prompt_txt):
            raise ValueError("chat template mismatch between prompt and full conversation")
        a_ids = tokenizer(full_txt[len(prompt_txt):], add_special_tokens=False)["input_ids"]
        ids = p_ids + a_ids
        if len(ids) > max_seq_len:
            skipped += 1
            continue
        feats.append({"input_ids": ids, "labels": [-100] * len(p_ids) + a_ids, "length": len(ids)})
    return feats, skipped


class PadCollator:
    """LEFT-pads so every answer sits at the end of its row. The trainer then only computes
    logits for the last K positions (K = longest answer in the batch) instead of the whole
    sequence - with Qwen's 152K-token vocabulary that saves several GB of GPU memory."""

    def __init__(self, pad_id: int):
        self.pad_id = pad_id

    def __call__(self, batch):
        import torch
        n = max(len(b["input_ids"]) for b in batch)
        ids = torch.full((len(batch), n), self.pad_id, dtype=torch.long)
        lab = torch.full((len(batch), n), -100, dtype=torch.long)
        att = torch.zeros((len(batch), n), dtype=torch.long)
        keep = 1
        for i, b in enumerate(batch):
            L = len(b["input_ids"])
            ids[i, n - L:] = torch.tensor(b["input_ids"])
            lab[i, n - L:] = torch.tensor(b["labels"])
            att[i, n - L:] = 1
            keep = max(keep, sum(1 for x in b["labels"] if x != -100))
        pos = (att.cumsum(-1) - 1).clamp(min=0)
        return {"input_ids": ids, "labels": lab, "attention_mask": att, "position_ids": pos,
                "answer_len": torch.tensor(keep)}


class LengthGroupedSampler:
    """Shuffles, then sorts examples by length inside mega-batches of 64 so each batch holds
    similar lengths (much less padding); batch order is shuffled again. Deterministic per epoch."""

    def __init__(self, lengths: list[int], batch_size: int, seed: int = 42, mega: int = 64):
        self.lengths, self.bs, self.seed, self.mega, self.epoch = lengths, batch_size, seed, mega, 0

    def __len__(self):
        return len(self.lengths)

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __iter__(self):
        import random
        rng = random.Random(self.seed + self.epoch)
        idx = list(range(len(self.lengths)))
        rng.shuffle(idx)
        batches = []
        for s in range(0, len(idx), self.mega):
            chunk = sorted(idx[s:s + self.mega], key=lambda i: -self.lengths[i])
            batches += [chunk[j:j + self.bs] for j in range(0, len(chunk), self.bs)]
        rng.shuffle(batches)
        self.epoch += 1
        return iter([i for b in batches for i in b])


def make_trainer_cls():
    from transformers import Trainer

    class AnswerOnlyTrainer(Trainer):
        """Causal-LM loss on the answer tokens only, computing logits just for those positions."""

        def __init__(self, *a, **kw):
            super().__init__(*a, **kw)
            self.model_accepts_loss_kwargs = False  # we return a per-micro-batch mean; Trainer divides by GA

        def _get_train_sampler(self, *args, **kwargs):
            lengths = self.train_dataset["length"]
            return LengthGroupedSampler(list(lengths), self.args.per_device_train_batch_size, self.args.seed)

        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            import torch.nn.functional as F
            labels = inputs.pop("labels")
            k = int(inputs.pop("answer_len"))
            try:
                out = model(**inputs, logits_to_keep=k + 1, use_cache=False)
                logits = out.logits[:, :-1, :]
                tgt = labels[:, -k:]
            except TypeError:  # very old transformers: full logits
                out = model(**inputs, use_cache=False)
                logits = out.logits[:, :-1, :]
                tgt = labels[:, 1:]
            loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-100)
            return (loss, out) if return_outputs else loss

    return AnswerOnlyTrainer


def train_qlora(cfg, train_examples: list[dict], schema_for, out_dir: Path, ckpt_dir: Path) -> dict:
    import torch
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed

    from .generation import _dtype_kwargs

    set_seed(cfg.seed)
    out_dir, ckpt_dir = Path(out_dir), Path(ckpt_dir)
    tok = AutoTokenizer.from_pretrained(cfg.base_model)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    t0 = time.time()
    feats, skipped = build_features(train_examples, tok, schema_for, cfg.max_seq_len, cfg.max_prompt_tokens)
    lengths = [f["length"] for f in feats]
    print(f"[train] {len(feats)} examples ({skipped} skipped > {cfg.max_seq_len} tokens); "
          f"mean length {sum(lengths)/len(lengths):.0f} tokens, total {sum(lengths)/1e6:.2f}M tokens")
    ds = Dataset.from_list(feats)

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(cfg.base_model, quantization_config=bnb, device_map={"": 0},
                                                 **_dtype_kwargs(torch.float16))
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
    lora = LoraConfig(r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout, bias="none",
                      task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, lora)
    try:  # PEFT counts packed 4-bit weights correctly
        trainable, total = model.get_nb_trainable_parameters()
    except Exception:
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
    print(f"[train] trainable params: {trainable/1e6:.1f}M of {total/1e6:.0f}M ({100*trainable/total:.2f}%)")

    steps_per_epoch = math.ceil(len(ds) / (cfg.per_device_batch_size * cfg.grad_accum_steps))
    max_steps = cfg.quick_max_steps if cfg.quick else -1
    total_steps = max_steps if max_steps > 0 else math.ceil(steps_per_epoch * cfg.num_epochs)
    targs = dict(
        output_dir=str(ckpt_dir),
        per_device_train_batch_size=cfg.per_device_batch_size,
        gradient_accumulation_steps=cfg.grad_accum_steps,
        num_train_epochs=cfg.num_epochs,
        max_steps=max_steps,
        learning_rate=cfg.learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=max(1, int(cfg.warmup_ratio * total_steps)),
        logging_steps=10,
        save_steps=cfg.save_steps,
        save_total_limit=2,
        fp16=True,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        report_to="none",
        seed=cfg.seed,
        remove_unused_columns=False,
    )
    # keep only arguments this transformers version understands (APIs move between major versions)
    import dataclasses
    known = {f.name for f in dataclasses.fields(TrainingArguments)}
    dropped = [k for k in targs if k not in known]
    if dropped:
        print("[train] TrainingArguments: ignoring unsupported", dropped)
    args = TrainingArguments(**{k: v for k, v in targs.items() if k in known})
    trainer = make_trainer_cls()(model=model, args=args, train_dataset=ds, data_collator=PadCollator(tok.pad_token_id))
    resume = any(ckpt_dir.glob("checkpoint-*")) if ckpt_dir.exists() else False
    if resume:
        print("[train] resuming from last checkpoint in", ckpt_dir)
    torch.cuda.reset_peak_memory_stats()
    t1 = time.time()
    result = trainer.train(resume_from_checkpoint=True if resume else None)
    train_s = time.time() - t1

    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(out_dir))
    tok.save_pretrained(str(out_dir))
    stats = {
        "train_examples": len(feats), "skipped_too_long": skipped,
        "total_train_tokens": int(sum(lengths)), "mean_seq_len": sum(lengths) / len(lengths),
        "steps": int(trainer.state.global_step), "steps_per_epoch": steps_per_epoch,
        "epochs": cfg.num_epochs if not cfg.quick else round(trainer.state.global_step / steps_per_epoch, 3),
        "train_runtime_s": train_s, "prep_s": t1 - t0,
        "final_train_loss": float(result.training_loss),
        "trainable_params": int(trainable), "total_params": int(total),
        "peak_gpu_mem_gb": torch.cuda.max_memory_allocated() / 1e9,
        "gpu": torch.cuda.get_device_name(0),
        "lora": {"r": cfg.lora_r, "alpha": cfg.lora_alpha, "dropout": cfg.lora_dropout},
        "effective_batch": cfg.per_device_batch_size * cfg.grad_accum_steps, "lr": cfg.learning_rate,
    }
    (out_dir / "train_stats.json").write_text(json.dumps(stats, indent=2))
    (out_dir / "log_history.json").write_text(json.dumps(trainer.state.log_history, indent=2))
    del trainer, model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    return stats


def merge_adapter(base_model: str, adapter_dir: Path, merged_dir: Path) -> Path:
    """Fold LoRA weights into the base model (fp16) so it can be served by vLLM."""
    import gc

    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    from .generation import _dtype_kwargs

    merged_dir = Path(merged_dir)
    if (merged_dir / "config.json").exists():
        print("[merge] already merged:", merged_dir)
        return merged_dir
    base = AutoModelForCausalLM.from_pretrained(base_model, device_map={"": 0}, **_dtype_kwargs(torch.float16))
    model = PeftModel.from_pretrained(base, str(adapter_dir))
    model = model.merge_and_unload()
    model.save_pretrained(str(merged_dir), safe_serialization=True)
    AutoTokenizer.from_pretrained(str(adapter_dir)).save_pretrained(str(merged_dir))
    # Use the base model's original config/tokenizer files (same architecture): files written by a newer
    # transformers can be unreadable or misread by the (older) transformers inside the vLLM environment.
    try:
        import fnmatch
        import shutil
        from huggingface_hub import snapshot_download
        # config.json too: a config written by a newer transformers (e.g. rope settings moved into
        # `rope_parameters`) is silently misread by the older transformers inside vLLM (wrong RoPE theta).
        pats = ["tokenizer*", "vocab.json", "merges.txt", "special_tokens_map.json", "added_tokens.json",
                "generation_config.json", "config.json"]
        src = Path(snapshot_download(base_model, allow_patterns=pats))
        # the cached snapshot folder also holds the base *weights*: copy only the tokenizer files
        for f in src.iterdir():
            if f.is_file() and any(fnmatch.fnmatch(f.name, p) for p in pats):
                shutil.copy(f, merged_dir / f.name)
    except Exception as e:  # pragma: no cover
        print("[merge] could not copy original tokenizer files:", e)
    del model, base
    gc.collect()
    torch.cuda.empty_cache()
    diff = verify_merge(base_model, merged_dir)
    print(f"[merge] saved {merged_dir} (max |merged - base| on a LoRA-targeted weight = {diff:.4g})")
    return merged_dir


def verify_merge(base_model: str, merged_dir: Path, key: str = "model.layers.0.self_attn.q_proj.weight") -> float:
    """Sanity check: the merged weights must differ from the base weights."""
    from huggingface_hub import snapshot_download
    from safetensors import safe_open

    def load(folder: Path):
        for f in sorted(Path(folder).glob("*.safetensors")):
            with safe_open(str(f), framework="pt") as sf:
                if key in sf.keys():
                    return sf.get_tensor(key).float()
        raise KeyError(key)

    base_dir = Path(snapshot_download(base_model, allow_patterns=["*.safetensors", "*.json"]))
    diff = float((load(merged_dir) - load(base_dir)).abs().max())
    if diff == 0.0:
        raise RuntimeError("Merged model is identical to the base model - LoRA weights were not applied")
    return diff

In [ ]:
%%writefile /content/instaquery/src/instaquery/serving.py
"""Serve a model with vLLM (OpenAI-compatible API) from an isolated virtual env.

vLLM pins its own torch/CUDA build; installing it into a separate `uv` venv
keeps the training environment (transformers/peft/bitsandbytes) untouched.
"""

from __future__ import annotations

import os
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

VENV = Path("/content/vllm-env")


def vllm_python() -> Path:
    return VENV / "bin" / "python"


def install_vllm(version: str = "0.19.0", log_file: Path | None = None) -> bool:
    if vllm_python().exists():
        r = subprocess.run([str(vllm_python()), "-c", "import vllm;print(vllm.__version__)"],
                           capture_output=True, text=True)
        if r.returncode == 0:
            print("[vllm] already installed:", r.stdout.strip())
            return True
    t0 = time.time()
    log = open(log_file, "w") if log_file else subprocess.DEVNULL
    try:
        if shutil.which("uv") is None:
            subprocess.run(["pip", "install", "-q", "uv"], check=True)
        subprocess.run(["uv", "venv", str(VENV), "--python", "3.12", "--seed"], check=True, stdout=log, stderr=log)
        subprocess.run(["uv", "pip", "install", "--python", str(vllm_python()), f"vllm=={version}"],
                       check=True, stdout=log, stderr=log)
        print(f"[vllm] installed vllm=={version} in {time.time()-t0:.0f}s")
        return True
    except subprocess.CalledProcessError as e:
        print("[vllm] install failed:", e, "- see", log_file)
        return False
    finally:
        if log_file:
            log.close()


class VLLMServer:
    def __init__(self, model_path: str, served_name: str, port: int = 8000, gpu_mem: float = 0.85,
                 max_model_len: int = 4096, log_file: Path | None = None, extra_args: list[str] | None = None,
                 attention_backend: str | None = "TRITON_ATTN"):
        # T4 (sm75) has no FlashAttention-2; vLLM would pick FlashInfer, which JIT-compiles kernels.
        # The Triton backend needs no compilation step.
        self.attention_backend = attention_backend
        self.model_path, self.served_name, self.port = str(model_path), served_name, port
        self.gpu_mem, self.max_model_len = gpu_mem, max_model_len
        self.log_file = Path(log_file) if log_file else Path("/content/vllm_server.log")
        self.extra_args = extra_args or []
        self.proc: subprocess.Popen | None = None
        self.startup_s = None

    @property
    def base_url(self) -> str:
        return f"http://127.0.0.1:{self.port}/v1"

    def _healthy(self) -> bool:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{self.port}/health", timeout=2) as r:
                return r.status == 200
        except Exception:
            return False

    def start(self, timeout_s: int = 900) -> bool:
        cmd = [str(vllm_python()), "-m", "vllm.entrypoints.openai.api_server",
               "--model", self.model_path, "--served-model-name", self.served_name,
               "--port", str(self.port), "--dtype", "half",
               "--gpu-memory-utilization", str(self.gpu_mem),
               "--max-model-len", str(self.max_model_len), "--seed", "0", *self.extra_args]
        if self.attention_backend:
            cmd += ["--attention-backend", self.attention_backend]
        env = dict(os.environ)
        env.pop("PYTHONPATH", None)
        env["PATH"] = f"{VENV / 'bin'}:{env.get('PATH', '')}"   # ninja etc. from the venv
        env.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")
        self._log = open(self.log_file, "w")
        t0 = time.time()
        self.proc = subprocess.Popen(cmd, stdout=self._log, stderr=subprocess.STDOUT, env=env)
        while time.time() - t0 < timeout_s:
            if self.proc.poll() is not None:
                print("[vllm] server exited early; last log lines:\n" + self.tail())
                return False
            if self._healthy():
                self.startup_s = time.time() - t0
                print(f"[vllm] server ready in {self.startup_s:.0f}s at {self.base_url}")
                return True
            time.sleep(5)
        print("[vllm] timed out waiting for server; last log lines:\n" + self.tail())
        self.stop()
        return False

    def tail(self, n: int = 40) -> str:
        try:
            return "\n".join(self.log_file.read_text(errors="ignore").splitlines()[-n:])
        except Exception:
            return ""

    def stop(self):
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(timeout=60)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        self.proc = None
        try:
            self._log.close()
        except Exception:
            pass
        time.sleep(5)  # let the GPU memory be released

In [ ]:
%%writefile /content/instaquery/src/instaquery/safety.py
"""Guardrails for running model-generated SQL against a live database (used by the demo app).

Layers: (1) parse with sqlglot and allow a single read-only SELECT/WITH statement,
(2) open the database in read-only mode, (3) hard timeout and row cap.
"""

from __future__ import annotations

import re

_FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|alter|create|replace|attach|detach|pragma|vacuum|reindex)\b", re.I)


def check_sql(sql: str) -> tuple[bool, str]:
    s = (sql or "").strip().rstrip(";")
    if not s:
        return False, "empty query"
    if ";" in s:
        try:
            import sqlglot
            if len([x for x in sqlglot.parse(s, read="sqlite") if x is not None]) > 1:
                return False, "only one statement is allowed"
        except Exception:
            return False, "only one statement is allowed"
    try:
        import sqlglot
        from sqlglot import exp
        tree = sqlglot.parse_one(s, read="sqlite")
        if not isinstance(tree, (exp.Select, exp.Union, exp.Intersect, exp.Except, exp.With)) and \
                not (isinstance(tree, exp.Subquery)):
            return False, f"only read-only SELECT queries are allowed (got {type(tree).__name__})"
        for node in tree.walk():
            n = node[0] if isinstance(node, tuple) else node
            if isinstance(n, (exp.Insert, exp.Update, exp.Delete, exp.Drop, exp.Create, exp.Alter)):
                return False, "data-modifying statements are not allowed"
    except ImportError:
        if not re.match(r"^\s*(with|select)\b", s, re.I) or _FORBIDDEN.search(s):
            return False, "only read-only SELECT queries are allowed"
    except Exception as e:
        return False, f"could not parse SQL: {e}"
    return True, "ok"

In [ ]:
%%writefile /content/instaquery/src/instaquery/domain_db.py
"""Synthetic Indian quick-commerce database (dark stores, orders, inventory ...).

Used as an out-of-domain test: the fine-tuned model never saw this schema, so
execution accuracy here measures transfer to a realistic business database.
Generated deterministically (seed 42); all names/brands are fictional.
"""

from __future__ import annotations

import random
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

CITY_AREAS = {
    "Bengaluru": [("Koramangala", "560034"), ("Indiranagar", "560038"), ("HSR Layout", "560102"),
                  ("Whitefield", "560066"), ("Jayanagar", "560041"), ("Electronic City", "560100"),
                  ("Marathahalli", "560037")],
    "Mumbai": [("Andheri West", "400053"), ("Bandra West", "400050"), ("Powai", "400076"),
               ("Lower Parel", "400013"), ("Thane West", "400601"), ("Malad West", "400064")],
    "Delhi": [("Saket", "110017"), ("Dwarka", "110075"), ("Rohini", "110085"), ("Lajpat Nagar", "110024"),
              ("Karol Bagh", "110005")],
    "Hyderabad": [("Gachibowli", "500032"), ("Madhapur", "500081"), ("Banjara Hills", "500034"),
                  ("Kukatpally", "500072")],
    "Pune": [("Kothrud", "411038"), ("Baner", "411045"), ("Viman Nagar", "411014"), ("Hinjewadi", "411057")],
    "Chennai": [("Adyar", "600020"), ("Velachery", "600042"), ("T Nagar", "600017"), ("Anna Nagar", "600040")],
}
CITY_CODE = {"Bengaluru": "BLR", "Mumbai": "MUM", "Delhi": "DEL", "Hyderabad": "HYD", "Pune": "PUN", "Chennai": "CHE"}
EXTRA_STORES = {"Koramangala": 3, "Andheri West": 2, "Gachibowli": 2, "Saket": 2}  # same-pincode clusters

CATALOG = {  # category -> (perishable, [(item, base_price)], brands)
    "Fruits & Vegetables": (1, [("Organic Bananas (6 pcs)", 69), ("Strawberries 200g", 149), ("Onion 1kg", 45),
                                ("Tomato 500g", 32), ("Potato 1kg", 39), ("Coriander Bunch", 15),
                                ("Shimla Apple 1kg", 189), ("Pomegranate 500g", 129), ("Spinach 250g", 29),
                                ("Avocado (2 pcs)", 249), ("Lemon (4 pcs)", 25)], ["FreshFarm", "GreenLeaf"]),
    "Dairy & Eggs": (1, [("Toned Milk 500ml", 28), ("Full Cream Milk 1L", 68), ("Curd 400g", 45), ("Paneer 200g", 90),
                         ("Butter 100g", 58), ("Cheese Slices 200g", 140), ("Brown Eggs (6 pcs)", 72),
                         ("Greek Yogurt 100g", 50)], ["DailyDairy", "MooFresh"]),
    "Bakery": (1, [("Whole Wheat Bread", 50), ("Multigrain Bread", 60), ("Pav (6 pcs)", 35), ("Croissant (2 pcs)", 99),
                   ("Chocolate Muffin", 45)], ["BakeHouse"]),
    "Snacks": (0, [("Classic Salted Chips 52g", 20), ("Masala Peanuts 200g", 60), ("Nachos 150g", 99),
                   ("Aloo Bhujia 400g", 110), ("Protein Bar", 120), ("Dark Chocolate 100g", 175),
                   ("Butter Cookies 250g", 85)], ["Crunchies", "SnackSutra"]),
    "Beverages": (0, [("Cola 750ml", 40), ("Orange Juice 1L", 125), ("Cold Coffee 200ml", 60), ("Green Tea 25 bags", 165),
                      ("Coconut Water 1L", 99), ("Energy Drink 250ml", 125), ("Packaged Water 1L", 20)],
                  ["SipWell", "ChillBox"]),
    "Staples": (0, [("Basmati Rice 1kg", 145), ("Toor Dal 1kg", 165), ("Whole Wheat Atta 5kg", 265),
                    ("Sunflower Oil 1L", 155), ("Sugar 1kg", 48), ("Iodised Salt 1kg", 28), ("Poha 500g", 45),
                    ("Besan 500g", 70)], ["AnnaPurna", "GharKa"]),
    "Personal Care": (0, [("Anti-Dandruff Shampoo 180ml", 199), ("Herbal Toothpaste 150g", 99),
                          ("Bath Soap (4 pcs)", 160), ("Face Wash 100ml", 175), ("Deodorant 150ml", 225)],
                      ["GlowUp", "PureCare"]),
    "Household": (0, [("Dishwash Liquid 500ml", 105), ("Detergent Powder 1kg", 130), ("Floor Cleaner 1L", 189),
                      ("Garbage Bags (30 pcs)", 99), ("Toilet Cleaner 500ml", 95)], ["CleanPro"]),
    "Baby Care": (0, [("Baby Diapers M (20 pcs)", 399), ("Baby Wipes (72 pcs)", 179)], ["LittleOne"]),
    "Frozen": (0, [("Frozen Green Peas 500g", 95), ("Vanilla Ice Cream Tub 700ml", 225), ("Frozen Paratha (5 pcs)", 110),
                   ("Chicken Nuggets 400g", 245)], ["FrostBite"]),
}
FIRST = ["Aarav", "Vivaan", "Aditya", "Vihaan", "Arjun", "Sai", "Reyansh", "Krishna", "Ishaan", "Rohan", "Ananya",
         "Diya", "Aadhya", "Saanvi", "Pari", "Anika", "Navya", "Myra", "Riya", "Kavya", "Meera", "Neha", "Priya",
         "Rahul", "Karan", "Siddharth", "Nikhil", "Tanvi", "Pooja", "Sneha", "Harsh", "Yash", "Aisha", "Zoya", "Imran",
         "Farhan", "Gurpreet", "Simran", "Lakshmi", "Venkat"]
LAST = ["Sharma", "Verma", "Iyer", "Reddy", "Patel", "Nair", "Gupta", "Mehta", "Rao", "Singh", "Kulkarni", "Joshi",
        "Das", "Banerjee", "Menon", "Pillai", "Khan", "Shaikh", "Chopra", "Malhotra", "Agarwal", "Desai", "Naidu",
        "Bhat", "Kapoor"]

SCHEMA_SQL = """
CREATE TABLE dark_stores (
  store_id INTEGER PRIMARY KEY,
  store_name TEXT NOT NULL,
  city TEXT NOT NULL,
  area TEXT NOT NULL,
  pincode TEXT NOT NULL,
  opened_date TEXT NOT NULL,
  area_sqft INTEGER,
  daily_order_capacity INTEGER
);
CREATE TABLE products (
  product_id INTEGER PRIMARY KEY,
  product_name TEXT NOT NULL,
  category TEXT NOT NULL,
  brand TEXT NOT NULL,
  mrp REAL NOT NULL,
  selling_price REAL NOT NULL,
  is_perishable INTEGER NOT NULL
);
CREATE TABLE customers (
  customer_id INTEGER PRIMARY KEY,
  customer_name TEXT NOT NULL,
  city TEXT NOT NULL,
  pincode TEXT NOT NULL,
  signup_date TEXT NOT NULL,
  is_plus_member INTEGER NOT NULL
);
CREATE TABLE delivery_partners (
  partner_id INTEGER PRIMARY KEY,
  partner_name TEXT NOT NULL,
  store_id INTEGER NOT NULL,
  vehicle_type TEXT NOT NULL,
  rating REAL,
  joined_date TEXT,
  FOREIGN KEY (store_id) REFERENCES dark_stores(store_id)
);
CREATE TABLE orders (
  order_id INTEGER PRIMARY KEY,
  customer_id INTEGER NOT NULL,
  store_id INTEGER NOT NULL,
  partner_id INTEGER,
  order_time TEXT NOT NULL,
  status TEXT NOT NULL,
  payment_method TEXT NOT NULL,
  delivery_minutes INTEGER,
  order_value REAL NOT NULL,
  discount REAL NOT NULL,
  FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
  FOREIGN KEY (store_id) REFERENCES dark_stores(store_id),
  FOREIGN KEY (partner_id) REFERENCES delivery_partners(partner_id)
);
CREATE TABLE order_items (
  order_id INTEGER NOT NULL,
  product_id INTEGER NOT NULL,
  quantity INTEGER NOT NULL,
  unit_price REAL NOT NULL,
  PRIMARY KEY (order_id, product_id),
  FOREIGN KEY (order_id) REFERENCES orders(order_id),
  FOREIGN KEY (product_id) REFERENCES products(product_id)
);
CREATE TABLE inventory (
  store_id INTEGER NOT NULL,
  product_id INTEGER NOT NULL,
  stock_qty INTEGER NOT NULL,
  reorder_level INTEGER NOT NULL,
  PRIMARY KEY (store_id, product_id),
  FOREIGN KEY (store_id) REFERENCES dark_stores(store_id),
  FOREIGN KEY (product_id) REFERENCES products(product_id)
);
"""


def _d(rng: random.Random, start: str, end: str) -> datetime:
    a, b = datetime.fromisoformat(start), datetime.fromisoformat(end)
    return a + timedelta(seconds=rng.randint(0, int((b - a).total_seconds())))


def build_quickcommerce_db(path: str | Path, n_customers: int = 3000, n_orders: int = 25000, seed: int = 42) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    rng = random.Random(seed)
    conn = sqlite3.connect(path)
    conn.executescript(SCHEMA_SQL)

    # ---- stores
    stores = []
    sid = 1
    for city, areas in CITY_AREAS.items():
        for area, pin in areas:
            for k in range(1, 1 + EXTRA_STORES.get(area, 1)):
                name = f"{CITY_CODE[city]}-{area.replace(' ', '')}-{k:02d}"
                opened = _d(rng, "2022-01-01", "2025-12-31").date().isoformat()
                stores.append((sid, name, city, area, pin, opened, rng.randrange(2500, 6001, 100),
                               rng.randrange(600, 2001, 50)))
                sid += 1
    conn.executemany("INSERT INTO dark_stores VALUES (?,?,?,?,?,?,?,?)", stores)

    # ---- products (product names are unique: brand + item)
    products, pid = [], 1
    for cat, (perish, items, brands) in CATALOG.items():
        for item, base in items:
            for brand in brands[: 1 + (rng.random() < 0.45)]:
                mrp = round(base * rng.uniform(0.95, 1.15))
                sp = round(mrp * rng.uniform(0.80, 0.98), 2)
                products.append((pid, f"{brand} {item}", cat, brand, float(mrp), sp, perish))
                pid += 1
    conn.executemany("INSERT INTO products VALUES (?,?,?,?,?,?,?)", products)
    never_ordered = {p[0] for p in rng.sample(products, 5)}
    orderable = [p for p in products if p[0] not in never_ordered]
    weights = [3.0 if ("Bananas" in p[1] or "Milk" in p[1] or "Bread" in p[1] or "Onion" in p[1]) else 1.0
               for p in orderable]

    # ---- customers
    custs = []
    city_list = list(CITY_AREAS)
    for cid in range(1, n_customers + 1):
        city = rng.choices(city_list, weights=[30, 25, 18, 12, 8, 7])[0]
        area, pin = rng.choice(CITY_AREAS[city])
        custs.append((cid, f"{rng.choice(FIRST)} {rng.choice(LAST)}", city, pin,
                      _d(rng, "2024-01-01", "2026-06-15").date().isoformat(), int(rng.random() < 0.25)))
    conn.executemany("INSERT INTO customers VALUES (?,?,?,?,?,?)", custs)

    # ---- delivery partners
    partners, ptid = [], 1
    by_store_partners: dict[int, list[int]] = {}
    for s in stores:
        for _ in range(rng.randint(6, 10)):
            vt = rng.choices(["bike", "scooter", "ev_scooter", "bicycle"], weights=[45, 25, 22, 8])[0]
            partners.append((ptid, f"{rng.choice(FIRST)} {rng.choice(LAST)}", s[0], vt,
                             round(rng.uniform(3.4, 5.0), 1), _d(rng, "2023-01-01", "2026-05-31").date().isoformat()))
            by_store_partners.setdefault(s[0], []).append(ptid)
            ptid += 1
    conn.executemany("INSERT INTO delivery_partners VALUES (?,?,?,?,?,?)", partners)

    # ---- orders + items
    stores_by_pin: dict[str, list[int]] = {}
    stores_by_city: dict[str, list[int]] = {}
    slow_store = {s[0]: rng.uniform(-2, 6) for s in stores}
    for s in stores:
        stores_by_pin.setdefault(s[4], []).append(s[0])
        stores_by_city.setdefault(s[2], []).append(s[0])
    hours = list(range(24))
    hour_w = [1, 0.5, 0.3, 0.2, 0.2, 0.5, 2, 5, 8, 7, 5, 4, 4, 4, 3, 3, 4, 5, 8, 10, 9, 7, 4, 2]
    banana = next(p for p in orderable if "Organic Bananas" in p[1])
    berries = next(p for p in orderable if "Strawberries" in p[1])
    orders, items = [], []
    plus = {c[0]: c[5] for c in custs}
    for oid in range(1001, 1001 + n_orders):
        c = rng.choice(custs)
        cands = stores_by_pin.get(c[3]) or stores_by_city[c[2]]
        st = rng.choice(cands)
        day = _d(rng, "2026-01-01", "2026-06-30").date()
        hr = rng.choices(hours, weights=hour_w)[0]
        ts = datetime(day.year, day.month, day.day, hr, rng.randint(0, 59), rng.randint(0, 59))
        status = rng.choices(["delivered", "cancelled", "returned"], weights=[92, 6, 2])[0]
        pay = rng.choices(["UPI", "Card", "COD", "Wallet"], weights=[55, 20, 15, 10])[0]
        n_items = rng.choices([1, 2, 3, 4, 5, 6], weights=[15, 25, 25, 18, 10, 7])[0]
        basket = {}
        if rng.random() < 0.08:  # bananas + strawberries are frequently bought together
            basket[banana[0]] = banana
            basket[berries[0]] = berries
        while len(basket) < n_items:
            p = rng.choices(orderable, weights=weights)[0]
            basket[p[0]] = p
        value = 0.0
        for p in basket.values():
            q = rng.choices([1, 2, 3], weights=[70, 22, 8])[0]
            items.append((oid, p[0], q, p[5]))
            value += q * p[5]
        value = round(value, 2)
        discount = round(value * rng.choice([0.0, 0.05, 0.10]), 2) if plus[c[0]] else 0.0
        if status == "cancelled":
            mins, partner = None, None
        else:
            peak = 3 if hr in (8, 9, 18, 19, 20) else 0
            mins = int(max(5, min(45, rng.gauss(12 + peak + slow_store[st], 4))))
            partner = rng.choice(by_store_partners[st])
        orders.append((oid, c[0], st, partner, ts.strftime("%Y-%m-%d %H:%M:%S"), status, pay, mins, value, discount))
    conn.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?,?,?,?,?)", orders)
    conn.executemany("INSERT INTO order_items VALUES (?,?,?,?)", items)

    # ---- inventory
    inv = []
    for s in stores:
        for p in products:
            reorder = rng.randrange(10, 41, 5)
            stock = 0 if rng.random() < 0.04 else rng.randint(0, 220)
            inv.append((s[0], p[0], stock, reorder))
    conn.executemany("INSERT INTO inventory VALUES (?,?,?,?)", inv)
    conn.commit()
    conn.close()
    return path

In [ ]:
%%writefile /content/instaquery/src/instaquery/pipeline.py
"""Orchestration used by the Colab notebook: data, prompt building, cached evaluation runs."""

from __future__ import annotations

import contextlib
import gc
import json
import random
import time
from pathlib import Path

from . import spider_data
from .domain_db import build_quickcommerce_db
from .generation import HFGenerator, OpenAIGenerator, load_hf_model
from .prompts import build_messages, extract_sql
from .retrieval import ExampleRetriever, pick_fixed_shots
from .schema import serialize_schema
from .serving import VLLMServer
from .sql_eval import score

DOMAIN_DB_ID = "quickcommerce"


class Workspace:
    def __init__(self, cfg, eval_json: Path):
        self.cfg = cfg
        self.eval_json = Path(eval_json)
        self.tokenizer = None
        self.retriever = None
        self.backend = "hf"          # switched to "vllm" once the server is verified
        self.timings: dict[str, float] = {}

    # ------------------------------------------------------------------ data
    def prepare_data(self):
        cfg = self.cfg
        t0 = time.time()
        self.spider_root = spider_data.download_spider(cfg.spider_dir)
        train = [e.to_dict() for e in spider_data.load_train(self.spider_root)]
        dev = [e.to_dict() for e in spider_data.load_dev(self.spider_root)]
        build_quickcommerce_db(cfg.domain_db_path)
        domain = json.loads(self.eval_json.read_text())
        for i, d in enumerate(domain):
            d.update({"db_id": DOMAIN_DB_ID, "idx": i, "split": "quickcommerce", "hardness": d["difficulty"]})
        rng = random.Random(cfg.seed)
        if cfg.quick:
            train = rng.sample(train, cfg.quick_train_examples)
            dev = rng.sample(dev, cfg.quick_dev_examples)
        self.train, self.dev, self.domain = train, dev, domain
        self.timings["data_prep_s"] = time.time() - t0
        print(f"[data] train={len(train)} dev={len(dev)} quickcommerce={len(domain)}")
        return self

    def db_lookup(self, db_id: str) -> Path:
        if db_id == DOMAIN_DB_ID:
            return Path(self.cfg.domain_db_path)
        return spider_data.db_path(self.spider_root, db_id)

    def schema_for(self, db_id: str, with_values: bool = True) -> str:
        return serialize_schema(str(self.db_lookup(db_id)),
                                self.cfg.sample_values_per_column if with_values else 0)

    def load_tokenizer(self):
        from transformers import AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.cfg.base_model)
        return self.tokenizer

    def build_retriever(self):
        if self.retriever is None:
            self.retriever = ExampleRetriever(self.train, self.cfg.retriever_model, device="cpu")
            print("[retriever]", self.retriever.method)
        return self.retriever

    # --------------------------------------------------------------- prompts
    def messages_for(self, examples: list[dict], shots: str = "zero") -> list[list[dict]]:
        """shots in {'zero', 'fixed', 'rag'}"""
        tok = self.tokenizer or self.load_tokenizer()
        if shots == "fixed":
            fixed = pick_fixed_shots(self.train, self.cfg.n_shots)
            shot_lists = [fixed] * len(examples)
        elif shots == "rag":
            r = self.build_retriever()
            shot_lists = r.retrieve_many([e["question"] for e in examples], self.cfg.n_shots,
                                         exclude_db=[e["db_id"] for e in examples])
        else:
            shot_lists = [None] * len(examples)
        out = []
        for ex, sh in zip(examples, shot_lists):
            msgs = build_messages(self.schema_for(ex["db_id"], True), ex["question"], sh)
            n = len(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)) // 3
            if n > self.cfg.max_prompt_tokens:  # cheap char estimate first, exact count only if close
                ids = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True),
                          add_special_tokens=False)["input_ids"]
                if len(ids) > self.cfg.max_prompt_tokens:
                    msgs = build_messages(self.schema_for(ex["db_id"], False), ex["question"], sh)
            out.append(msgs)
        return out

    # ------------------------------------------------------------ backends
    @contextlib.contextmanager
    def generator(self, model_path: str, backend: str | None = None, quant_4bit: bool = False,
                  tag: str = "model"):
        backend = backend or self.backend
        cfg = self.cfg
        if backend == "vllm":
            srv = VLLMServer(model_path, cfg.served_model_name, cfg.vllm_port, cfg.vllm_gpu_mem,
                             cfg.vllm_max_model_len, log_file=Path(cfg.output_dir) / "logs" / f"vllm_{tag}.log")
            Path(cfg.output_dir, "logs").mkdir(parents=True, exist_ok=True)
            if not srv.start():
                raise RuntimeError("vLLM server failed to start")
            self.timings[f"vllm_startup_{tag}_s"] = srv.startup_s
            try:
                # plain greedy decoding (override the model's default repetition penalty, same as HF path)
                yield OpenAIGenerator(srv.base_url, cfg.served_model_name, max_new_tokens=cfg.max_new_tokens,
                                      concurrency=64, extra_body={"repetition_penalty": 1.0, "top_p": 1.0})
            finally:
                srv.stop()
        else:
            import torch
            model, tok = load_hf_model(model_path, quant_4bit=quant_4bit)
            try:
                gen = HFGenerator(model, tok, cfg.hf_eval_batch_size, cfg.max_new_tokens)
                gen.memory_gb = model.get_memory_footprint() / 1e9
                yield gen
            finally:
                del model
                gc.collect()
                torch.cuda.empty_cache()

    # ------------------------------------------------------------ evaluation
    def run_variant(self, gen, variant: str, dataset: str, shots: str = "zero", force: bool = False,
                    extra_meta: dict | None = None) -> dict:
        cfg = self.cfg
        mfile = Path(cfg.output_dir) / "metrics" / f"{dataset}__{variant}.json"
        pfile = Path(cfg.output_dir) / "predictions" / dataset / f"{variant}.jsonl"
        if mfile.exists() and not force:
            print(f"[eval] cached: {dataset}/{variant}")
            return json.loads(mfile.read_text())
        examples = self.dev if dataset == "spider_dev" else self.domain
        msgs = self.messages_for(examples, shots)
        t0 = time.time()
        outs = gen.generate(msgs, desc=f"{dataset}/{variant}")
        gen_s = time.time() - t0
        failed = sum(o.text.startswith("-- request failed") for o in outs)
        if failed > 0.2 * len(outs):
            raise RuntimeError(f"{failed}/{len(outs)} generation requests failed: {outs[0].text[:300]}")
        preds = [extract_sql(o.text) for o in outs]
        extra = [{"raw": o.text, "latency_s": o.latency_s, "prompt_tokens": o.prompt_tokens,
                  "completion_tokens": o.completion_tokens} for o in outs]
        sc = score(examples, preds, self.db_lookup, cfg.sql_timeout_s, extra,
                   keep_distinct=(dataset != "spider_dev"))
        pfile.parent.mkdir(parents=True, exist_ok=True)
        with open(pfile, "w") as f:
            for r in sc.records:
                f.write(json.dumps(r) + "\n")
        n = len(examples)
        metrics = {
            "variant": variant, "dataset": dataset, "shots": shots, "n": n,
            "execution_accuracy": sc.ex, "valid_sql_rate": sc.exec_rate,
            "by_difficulty": {k: {"ex": v[0], "n": v[1]} for k, v in sc.by("hardness").items()},
            "error_breakdown": sc.error_counts(),
            "generation_time_s": gen_s, "examples_per_s": n / gen_s if gen_s else None,
            "avg_prompt_tokens": sum(e["prompt_tokens"] for e in extra) / n,
            "avg_completion_tokens": sum(e["completion_tokens"] for e in extra) / n,
            "backend": type(gen).__name__,
        }
        if hasattr(gen, "memory_gb"):
            metrics["model_memory_gb"] = gen.memory_gb
        if extra_meta:
            metrics.update(extra_meta)
        mfile.parent.mkdir(parents=True, exist_ok=True)
        mfile.write_text(json.dumps(metrics, indent=2))
        print(f"[eval] {dataset}/{variant}: EX={sc.ex:.3f} valid={sc.exec_rate:.3f} "
              f"({n} ex, {gen_s:.0f}s)")
        return metrics

    # ------------------------------------------------------- plans + bench
    def run_plan(self, model_path: str, plan: list[tuple[str, str, str]], tag: str,
                 backend: str | None = None, quant_4bit: bool = False, after=None) -> list[dict]:
        """plan: [(variant, dataset, shots)]. Falls back from vLLM to HF if the server fails.
        `after(gen)` runs inside the same model session (used for the throughput benchmark)."""
        cfg = self.cfg
        todo = [p for p in plan if not (Path(cfg.output_dir) / "metrics" / f"{p[1]}__{p[0]}.json").exists()]
        if not todo and after is None:
            return [json.loads((Path(cfg.output_dir) / "metrics" / f"{d}__{v}.json").read_text()) for v, d, _ in plan]
        backend = backend or self.backend
        try:
            with self.generator(model_path, backend=backend, quant_4bit=quant_4bit, tag=tag) as gen:
                res = [self.run_variant(gen, v, d, s) for v, d, s in plan]
                if after is not None:
                    after(gen)
                return res
        except RuntimeError as e:
            if backend != "vllm":
                raise
            print(f"[eval] vLLM unavailable ({e}); falling back to Hugging Face generation")
            self.backend = "hf"
            return self.run_plan(model_path, plan, tag, backend="hf", quant_4bit=quant_4bit, after=after)

    def bench_messages(self) -> list[list[dict]]:
        n = min(self.cfg.bench_requests if not self.cfg.quick else 50, len(self.dev))
        return self.messages_for(self.dev[:n], "zero")

    def bench_vllm(self, gen) -> list[dict]:
        msgs = self.bench_messages()
        gen.generate(msgs[:8], concurrency=8)  # warm-up
        rows = []
        for c in self.cfg.bench_concurrency:
            r = gen.timed_batch(msgs, concurrency=c)
            print(f"[bench] vLLM concurrency={c}: {r['req_per_s']:.2f} req/s, "
                  f"{r['output_tok_per_s']:.0f} tok/s, p50 {r['p50_latency_s']:.2f}s")
            rows.append(r)
        return rows

    def bench_hf(self, model_path: str, single: int = 20) -> dict:
        import torch
        msgs = self.bench_messages()
        out = {}
        with self.generator(model_path, backend="hf", tag="hf_bench") as gen:
            gen.generate(msgs[:4])  # warm-up
            t0 = time.perf_counter()
            res = gen.generate(msgs, desc="hf batched")
            wall = time.perf_counter() - t0
            toks = sum(o.completion_tokens for o in res)
            out["hf_batched"] = {"batch_size": gen.batch_size, "requests": len(msgs), "wall_s": wall,
                                 "req_per_s": len(msgs) / wall, "output_tok_per_s": toks / wall}
            lat = sorted(gen.generate_one_timed(m).latency_s for m in msgs[:single])
            out["hf_single"] = {"requests": len(lat), "p50_latency_s": lat[len(lat) // 2],
                                "mean_latency_s": sum(lat) / len(lat)}
            out["hf_fp16_memory_gb"] = gen.memory_gb
        print(f"[bench] HF batched: {out['hf_batched']['req_per_s']:.2f} req/s; "
              f"single-request p50 {out['hf_single']['p50_latency_s']:.2f}s")
        torch.cuda.empty_cache()
        return out

In [ ]:
%%writefile /content/instaquery/src/instaquery/report.py
"""Collect metrics -> summary tables, charts, README results section and CV bullets."""

from __future__ import annotations

import json
from pathlib import Path

# validated categorical palette (light surface) - see README "charts"
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, AQUA, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#a3a29d"

LABELS = {
    "base_zeroshot": "Base, zero-shot",
    "base_fewshot": "Base, 3 fixed examples",
    "base_rag": "Base + RAG (3 retrieved examples)",
    "finetuned": "Fine-tuned (QLoRA)",
    "finetuned_rag": "Fine-tuned + RAG",
    "finetuned_nf4": "Fine-tuned, 4-bit NF4",
    "api_zeroshot": "Paid API model, zero-shot",
}


# one colour per variant family, identical in every chart
VARIANT_COLOR = {"base_zeroshot": MUTED, "base_fewshot": MUTED, "base_rag": AQUA, "finetuned": BLUE,
                 "finetuned_rag": BLUE, "finetuned_nf4": BLUE, "api_zeroshot": ORANGE}


def load_metrics(out_dir: Path) -> list[dict]:
    return [json.loads(p.read_text()) for p in sorted((Path(out_dir) / "metrics").glob("*.json"))]


def _style(ax):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=9)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


DATASET_LABELS = {"spider_dev": "Spider dev (1,034)", "quickcommerce": "Quick-commerce (50)"}


def summary_table(metrics: list[dict]) -> str:
    rows = ["| Dataset | Variant | Execution accuracy | Valid SQL | Examples | Gen. time |",
            "|---|---|---|---|---|---|"]
    order = list(LABELS)
    dorder = ["spider_dev", "quickcommerce"]
    for m in sorted(metrics, key=lambda m: (dorder.index(m["dataset"]) if m["dataset"] in dorder else 9,
                                            order.index(m["variant"]) if m["variant"] in order else 99)):
        rows.append(f"| {DATASET_LABELS.get(m['dataset'], m['dataset'])} | {LABELS.get(m['variant'], m['variant'])} | "
                    f"**{100*m['execution_accuracy']:.1f}%** | {100*m['valid_sql_rate']:.1f}% | {m['n']} | "
                    f"{m['generation_time_s']:.0f}s |")
    return "\n".join(rows)


def chart_ex_by_variant(metrics: list[dict], path: Path):
    import matplotlib.pyplot as plt
    datasets = [d for d in ("spider_dev", "quickcommerce") if any(m["dataset"] == d for m in metrics)]
    fig, axes = plt.subplots(1, len(datasets), figsize=(6.2 * len(datasets), 4.2), facecolor=SURFACE)
    axes = [axes] if len(datasets) == 1 else axes
    titles = {"spider_dev": "Spider dev (1,034 questions, 20 unseen databases)",
              "quickcommerce": "Quick-commerce DB (50 questions, unseen schema)"}
    for ax, d in zip(axes, datasets):
        ms = [m for m in metrics if m["dataset"] == d and m["variant"] in LABELS]
        ms.sort(key=lambda m: list(LABELS).index(m["variant"]))
        names = [LABELS[m["variant"]] for m in ms]
        vals = [100 * m["execution_accuracy"] for m in ms]
        cols = [VARIANT_COLOR.get(m["variant"], MUTED) for m in ms]
        y = range(len(ms))
        ax.barh(list(y), vals, color=cols, height=0.62, edgecolor=SURFACE, linewidth=2)
        for yi, v in zip(y, vals):
            ax.text(v + 1, yi, f"{v:.1f}%", va="center", fontsize=9, color=INK)
        ax.set_yticks(list(y), names, fontsize=9, color=INK)
        ax.invert_yaxis()
        ax.set_xlim(0, 100)
        ax.set_xlabel("Execution accuracy (%)", color=INK2, fontsize=9)
        ax.set_title(titles.get(d, d), fontsize=10, color=INK, loc="left")
        _style(ax)
        ax.xaxis.grid(True, color=GRID, linewidth=0.8)
        ax.yaxis.grid(False)
    fig.tight_layout()
    fig.savefig(path, dpi=160, facecolor=SURFACE)
    plt.close(fig)


def chart_by_difficulty(metrics: list[dict], path: Path, dataset: str = "spider_dev",
                        variants=("base_zeroshot", "base_rag", "finetuned")):
    import matplotlib.pyplot as plt
    levels = ["easy", "medium", "hard", "extra"]
    ms = {m["variant"]: m for m in metrics if m["dataset"] == dataset}
    variants = [v for v in variants if v in ms]
    if not variants:
        return
    levels = [l for l in levels if any(l in ms[v]["by_difficulty"] for v in variants)]
    fig, ax = plt.subplots(figsize=(7.5, 4), facecolor=SURFACE)
    w = 0.8 / len(variants)
    for i, v in enumerate(variants):
        vals = [100 * ms[v]["by_difficulty"].get(l, {}).get("ex", 0) for l in levels]
        xs = [j + (i - (len(variants) - 1) / 2) * w for j in range(len(levels))]
        ax.bar(xs, vals, width=w, color=VARIANT_COLOR.get(v, MUTED), label=LABELS[v], edgecolor=SURFACE, linewidth=2)
        for x, val in zip(xs, vals):
            ax.text(x, val + 1, f"{val:.0f}", ha="center", fontsize=8, color=INK2)
    counts = [ms[variants[0]]["by_difficulty"].get(l, {}).get("n", 0) for l in levels]
    ax.set_xticks(range(len(levels)), [f"{l}\n(n={c})" for l, c in zip(levels, counts)], color=INK)
    ax.set_ylabel("Execution accuracy (%)", color=INK2, fontsize=9)
    ax.set_ylim(0, 105)
    ax.set_title("Accuracy by Spider difficulty level", fontsize=10, color=INK, loc="left")
    ax.legend(frameon=False, fontsize=8, labelcolor=INK)
    _style(ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160, facecolor=SURFACE)
    plt.close(fig)


def chart_training_loss(log_history: list[dict], path: Path):
    import matplotlib.pyplot as plt
    pts = [(h["step"], h["loss"]) for h in log_history if "loss" in h and "step" in h]
    if not pts:
        return
    fig, ax = plt.subplots(figsize=(7, 3.6), facecolor=SURFACE)
    ax.plot([p[0] for p in pts], [p[1] for p in pts], color=BLUE, linewidth=2)
    ax.set_xlabel("Optimizer step", color=INK2, fontsize=9)
    ax.set_ylabel("Training loss (SQL tokens)", color=INK2, fontsize=9)
    ax.set_title("QLoRA training loss", fontsize=10, color=INK, loc="left")
    _style(ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160, facecolor=SURFACE)
    plt.close(fig)


def chart_throughput(bench: dict, path: Path):
    import matplotlib.pyplot as plt
    v = bench.get("vllm") or []
    if not v:
        return
    fig, ax = plt.subplots(figsize=(7, 3.8), facecolor=SURFACE)
    xs = [r["concurrency"] for r in v]
    ys = [r["req_per_s"] for r in v]
    ax.plot(xs, ys, color=BLUE, linewidth=2, marker="o", markersize=7, label="vLLM server (fp16)")
    for x, y in zip(xs, ys):
        ax.annotate(f"{y:.1f}", (x, y), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=8,
                    color=INK)
    hf = bench.get("hf_batched")
    if hf:
        ax.axhline(hf["req_per_s"], color=ORANGE, linewidth=2, linestyle="--",
                   label=f"HF transformers, batch {hf['batch_size']}")
    ax.set_xscale("log", base=2)
    ax.set_xticks(xs, [str(x) for x in xs])
    ax.set_ylim(bottom=0, top=max(ys) * 1.15)
    ax.set_xlabel("Concurrent requests", color=INK2, fontsize=9)
    ax.set_ylabel("Requests per second", color=INK2, fontsize=9)
    ax.set_title("Serving throughput on one T4 GPU", fontsize=10, color=INK, loc="left")
    ax.legend(frameon=False, fontsize=8, labelcolor=INK)
    _style(ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160, facecolor=SURFACE)
    plt.close(fig)


def error_analysis(out_dir: Path, dataset: str = "spider_dev", a: str = "base_zeroshot",
                   b: str = "finetuned", k: int = 5) -> str:
    pa = Path(out_dir) / "predictions" / dataset / f"{a}.jsonl"
    pb = Path(out_dir) / "predictions" / dataset / f"{b}.jsonl"
    if not (pa.exists() and pb.exists()):
        return ""
    ra = [json.loads(x) for x in pa.read_text().splitlines()]
    rb = [json.loads(x) for x in pb.read_text().splitlines()]
    fixed = [(x, y) for x, y in zip(ra, rb) if not x["correct"] and y["correct"]]
    broke = [(x, y) for x, y in zip(ra, rb) if x["correct"] and not y["correct"]]
    lines = [f"### What fine-tuning changed ({dataset})", "",
             f"* Questions the base model got wrong and the fine-tuned model got right: **{len(fixed)}**",
             f"* Questions the base model got right and the fine-tuned model got wrong: **{len(broke)}**", "",
             "| Error type | " + LABELS.get(a, a) + " | " + LABELS.get(b, b) + " |", "|---|---|---|"]
    from collections import Counter
    ca, cb = Counter(r["category"] for r in ra), Counter(r["category"] for r in rb)
    for cat in sorted(set(ca) | set(cb), key=lambda c: -(ca[c] + cb[c])):
        if cat != "correct":
            lines.append(f"| {cat} | {ca[cat]} | {cb[cat]} |")
    lines += ["", "Examples fixed by fine-tuning:", ""]
    for x, y in fixed[:k]:
        lines += [f"* **Q:** {x['question']}  ", f"  base: `{x['pred'][:220]}` ({x['category']})  ",
                  f"  fine-tuned: `{y['pred'][:220]}`"]
    return "\n".join(lines)


def cv_bullets(metrics: list[dict], train_stats: dict | None, bench: dict | None, model_name: str) -> str:
    get = {(m["dataset"], m["variant"]): m for m in metrics}
    def ex(d, v):
        m = get.get((d, v))
        return 100 * m["execution_accuracy"] if m else None
    b0, br, ft = ex("spider_dev", "base_zeroshot"), ex("spider_dev", "base_rag"), ex("spider_dev", "finetuned")
    q0, qft = ex("quickcommerce", "base_zeroshot"), ex("quickcommerce", "finetuned")
    nf4 = get.get(("spider_dev", "finetuned_nf4"))
    api = ex("spider_dev", "api_zeroshot")
    out = []
    if ft is not None and b0 is not None:
        tr = train_stats or {}
        out.append(f"Fine-tuned {model_name} for Text-to-SQL with QLoRA (4-bit NF4 + LoRA r={tr.get('lora', {}).get('r', 16)}, "
                   f"{tr.get('trainable_params', 0)/1e6:.1f}M trainable params) on {tr.get('train_examples', 0):,} Spider "
                   f"examples on a single T4 GPU in {tr.get('train_runtime_total_s', tr.get('train_runtime_s', 0))/60:.0f} min, raising execution accuracy on "
                   f"1,034 unseen-database questions from {b0:.1f}% to {ft:.1f}% (+{ft-b0:.1f} pts).")
    if br is not None and ft is not None:
        line = (f"Benchmarked zero-shot, fixed few-shot, retrieval-augmented few-shot (dense retrieval of similar "
                f"examples) and fine-tuned variants with an execution-based evaluator; RAG reached {br:.1f}% vs "
                f"{ft:.1f}% for the fine-tuned model")
        if api is not None:
            line += f", and a paid API model reached {api:.1f}%"
        out.append(line + ".")
    if q0 is not None and qft is not None:
        out.append(f"Built a synthetic quick-commerce SQLite database (7 tables, 25K orders) with 50 hand-verified "
                   f"business questions to test transfer to an unseen schema: {q0:.0f}% -> {qft:.0f}% execution accuracy.")
    if bench and bench.get("vllm"):
        best = max(bench["vllm"], key=lambda r: r["req_per_s"])
        hf = bench.get("hf_batched", {})
        speed = f" ({best['req_per_s']/hf['req_per_s']:.1f}x the throughput of batched HF generation)" if hf else ""
        out.append(f"Served the merged model with vLLM behind an OpenAI-compatible API at {best['req_per_s']:.1f} req/s "
                   f"and {best['output_tok_per_s']:.0f} output tokens/s at concurrency {best['concurrency']}{speed}.")
    if nf4 and ft is not None:
        out.append(f"Quantized the fine-tuned model to 4-bit NF4: {100*nf4['execution_accuracy']:.1f}% execution accuracy "
                   f"({100*nf4['execution_accuracy']-ft:+.1f} pts) in {nf4.get('model_memory_gb', 0):.2f} GB of GPU memory.")
    return "\n".join(f"- {x}" for x in out)


def build_readme(template: str, cfg, metrics: list[dict], train_stats: dict | None, bench: dict | None,
                 error_md: str, official_em: dict | None = None, findings_md: str = "",
                 engineering_md: str = "") -> str:
    get = {(m["dataset"], m["variant"]): m for m in metrics}
    b0, ft = get.get(("spider_dev", "base_zeroshot")), get.get(("spider_dev", "finetuned"))
    headline = ""
    if b0 and ft:
        headline = (f"> **Spider dev execution accuracy: {100*b0['execution_accuracy']:.1f}% → "
                    f"{100*ft['execution_accuracy']:.1f}%** after QLoRA fine-tuning "
                    f"({100*(ft['execution_accuracy']-b0['execution_accuracy']):+.1f} pts) "
                    f"on {ft['n']:,} questions over databases never seen in training.")
    ts = train_stats or {}
    train_rows = [
        f"| Training examples | {ts.get('train_examples', 'n/a'):,} ({ts.get('total_train_tokens', 0)/1e6:.2f}M tokens) |"
        if ts else "| Training examples | n/a |",
        f"| Trainable parameters | {ts.get('trainable_params', 0)/1e6:.1f}M of {ts.get('total_params', 0)/1e6:.0f}M "
        f"({100*ts.get('trainable_params', 0)/max(1, ts.get('total_params', 1)):.2f}%) |",
        f"| Steps / epochs | {ts.get('steps', 'n/a')} / {ts.get('epochs', 'n/a')} |",
        f"| Training time | {ts.get('train_runtime_total_s', ts.get('train_runtime_s', 0))/60:.0f} min on {ts.get('gpu', 'GPU')}"
        f"{' (' + ts['train_runtime_note'] + ')' if ts.get('train_runtime_note') else ''} |",
        f"| Peak GPU memory | {ts.get('peak_gpu_mem_gb', 0):.1f} GB |",
        f"| Final training loss | {ts.get('final_train_loss', float('nan')):.3f} |",
    ]
    if official_em:
        for k, v in official_em.items():
            train_rows.append(f"| Official exact-set-match, {LABELS.get(k, k)} | {100*v.get('all', 0):.1f}% |")
    bench_md = "_not run_"
    if bench and bench.get("vllm"):
        rows = ["| Setup | Requests/s | Output tokens/s | p50 latency |", "|---|---|---|---|"]
        hs = bench.get("hf_single")
        if hs:
            rows.append(f"| HF generate, 1 request at a time | {1/hs['mean_latency_s']:.2f} | - | {hs['p50_latency_s']:.2f}s |")
        hb = bench.get("hf_batched")
        if hb:
            rows.append(f"| HF generate, batch {hb['batch_size']} | {hb['req_per_s']:.2f} | {hb['output_tok_per_s']:.0f} | - |")
        for r in bench["vllm"]:
            rows.append(f"| vLLM, concurrency {r['concurrency']} | {r['req_per_s']:.2f} | {r['output_tok_per_s']:.0f} | "
                        f"{r['p50_latency_s']:.2f}s |")
        bench_md = "\n".join(rows)
    gpu = ts.get("gpu", "T4")
    rep = {"{{MODEL}}": cfg.base_model, "{{HEADLINE}}": headline, "{{RUN_MODE}}": cfg.run_mode, "{{GPU}}": gpu,
           "{{RESULTS_TABLE}}": summary_table(metrics), "{{TRAIN_TABLE}}": "\n".join(train_rows),
           "{{BENCH_TABLE}}": bench_md, "{{ERROR_ANALYSIS}}": error_md,
           "{{FINDINGS}}": findings_md, "{{ENGINEERING}}": engineering_md}
    for k, v in rep.items():
        template = template.replace(k, v)
    return template

In [ ]:
%%writefile /content/instaquery/data/quickcommerce_eval.json
[
  {"difficulty": "easy", "question": "How many dark stores are there in Bengaluru?",
   "query": "SELECT COUNT(*) FROM dark_stores WHERE city = 'Bengaluru'"},
  {"difficulty": "easy", "question": "List the names of all products in the 'Frozen' category.",
   "query": "SELECT product_name FROM products WHERE category = 'Frozen'"},
  {"difficulty": "easy", "question": "How many customers are Plus members?",
   "query": "SELECT COUNT(*) FROM customers WHERE is_plus_member = 1"},
  {"difficulty": "easy", "question": "What is the average rating of delivery partners whose vehicle type is 'ev_scooter'?",
   "query": "SELECT AVG(rating) FROM delivery_partners WHERE vehicle_type = 'ev_scooter'"},
  {"difficulty": "easy", "question": "How many orders were cancelled?",
   "query": "SELECT COUNT(*) FROM orders WHERE status = 'cancelled'"},
  {"difficulty": "easy", "question": "Which product has the highest MRP? Give its name.",
   "query": "SELECT product_name FROM products ORDER BY mrp DESC LIMIT 1"},
  {"difficulty": "easy", "question": "How many dark stores does each city have?",
   "query": "SELECT city, COUNT(*) FROM dark_stores GROUP BY city"},
  {"difficulty": "easy", "question": "List the names of dark stores that opened in 2025.",
   "query": "SELECT store_name FROM dark_stores WHERE strftime('%Y', opened_date) = '2025'"},
  {"difficulty": "easy", "question": "How many orders were paid by cash on delivery (payment method 'COD')?",
   "query": "SELECT COUNT(*) FROM orders WHERE payment_method = 'COD'"},
  {"difficulty": "easy", "question": "How many perishable products are there?",
   "query": "SELECT COUNT(*) FROM products WHERE is_perishable = 1"},
  {"difficulty": "easy", "question": "What is the average delivery time in minutes of delivered orders?",
   "query": "SELECT AVG(delivery_minutes) FROM orders WHERE status = 'delivered'"},
  {"difficulty": "easy", "question": "List all the different product categories.",
   "query": "SELECT DISTINCT category FROM products"},
  {"difficulty": "easy", "question": "Which pincodes have more than one dark store?",
   "query": "SELECT pincode FROM dark_stores GROUP BY pincode HAVING COUNT(*) > 1"},
  {"difficulty": "easy", "question": "How many products does each brand sell?",
   "query": "SELECT brand, COUNT(*) FROM products GROUP BY brand"},
  {"difficulty": "easy", "question": "What is the difference between the highest and the lowest selling price in the 'Beverages' category?",
   "query": "SELECT MAX(selling_price) - MIN(selling_price) FROM products WHERE category = 'Beverages'"},

  {"difficulty": "medium", "question": "What is the total order value of delivered orders for each city where the dark store is located?",
   "query": "SELECT s.city, SUM(o.order_value) FROM orders AS o JOIN dark_stores AS s ON o.store_id = s.store_id WHERE o.status = 'delivered' GROUP BY s.city"},
  {"difficulty": "medium", "question": "Which dark store received the most orders? Give the store name.",
   "query": "SELECT s.store_name FROM orders AS o JOIN dark_stores AS s ON o.store_id = s.store_id GROUP BY s.store_id ORDER BY COUNT(*) DESC LIMIT 1"},
  {"difficulty": "medium", "question": "How many orders were placed by customers who live in Mumbai?",
   "query": "SELECT COUNT(*) FROM orders AS o JOIN customers AS c ON o.customer_id = c.customer_id WHERE c.city = 'Mumbai'"},
  {"difficulty": "medium", "question": "What is the total quantity sold of 'FreshFarm Organic Bananas (6 pcs)' across all orders?",
   "query": "SELECT SUM(oi.quantity) FROM order_items AS oi JOIN products AS p ON oi.product_id = p.product_id WHERE p.product_name = 'FreshFarm Organic Bananas (6 pcs)'"},
  {"difficulty": "medium", "question": "List the names of delivery partners at the store 'BLR-Koramangala-01' with a rating above 4.5.",
   "query": "SELECT d.partner_name FROM delivery_partners AS d JOIN dark_stores AS s ON d.store_id = s.store_id WHERE s.store_name = 'BLR-Koramangala-01' AND d.rating > 4.5"},
  {"difficulty": "medium", "question": "What is the average order value for each payment method?",
   "query": "SELECT payment_method, AVG(order_value) FROM orders GROUP BY payment_method"},
  {"difficulty": "medium", "question": "Which product category has the most out-of-stock entries (stock quantity 0) in the inventory?",
   "query": "SELECT p.category FROM inventory AS i JOIN products AS p ON i.product_id = p.product_id WHERE i.stock_qty = 0 GROUP BY p.category ORDER BY COUNT(*) DESC LIMIT 1"},
  {"difficulty": "medium", "question": "How many orders were placed in March 2026?",
   "query": "SELECT COUNT(*) FROM orders WHERE strftime('%Y-%m', order_time) = '2026-03'"},
  {"difficulty": "medium", "question": "For each month (formatted as YYYY-MM), how many orders were delivered?",
   "query": "SELECT strftime('%Y-%m', order_time), COUNT(*) FROM orders WHERE status = 'delivered' GROUP BY strftime('%Y-%m', order_time)"},
  {"difficulty": "medium", "question": "What are the names of products that have never been ordered?",
   "query": "SELECT product_name FROM products WHERE product_id NOT IN (SELECT product_id FROM order_items)"},
  {"difficulty": "medium", "question": "How many customers have never placed an order?",
   "query": "SELECT COUNT(*) FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM orders)"},
  {"difficulty": "medium", "question": "For each vehicle type, what is the average delivery time of delivered orders handled by partners using that vehicle?",
   "query": "SELECT d.vehicle_type, AVG(o.delivery_minutes) FROM orders AS o JOIN delivery_partners AS d ON o.partner_id = d.partner_id WHERE o.status = 'delivered' GROUP BY d.vehicle_type"},
  {"difficulty": "medium", "question": "For each dark store in Delhi, how many orders were cancelled? Show the store name and the count.",
   "query": "SELECT s.store_name, COUNT(*) FROM orders AS o JOIN dark_stores AS s ON o.store_id = s.store_id WHERE s.city = 'Delhi' AND o.status = 'cancelled' GROUP BY s.store_id"},
  {"difficulty": "medium", "question": "What are the names of the 5 products with the highest sales revenue (quantity times unit price)?",
   "query": "SELECT p.product_name FROM order_items AS oi JOIN products AS p ON oi.product_id = p.product_id GROUP BY p.product_id ORDER BY SUM(oi.quantity * oi.unit_price) DESC LIMIT 5"},
  {"difficulty": "medium", "question": "How many units in total (sum of quantities) are in order 1001?",
   "query": "SELECT SUM(quantity) FROM order_items WHERE order_id = 1001"},
  {"difficulty": "medium", "question": "How many customers signed up in 2026 and are Plus members?",
   "query": "SELECT COUNT(*) FROM customers WHERE strftime('%Y', signup_date) = '2026' AND is_plus_member = 1"},
  {"difficulty": "medium", "question": "What is the average rating of delivery partners in each city?",
   "query": "SELECT s.city, AVG(d.rating) FROM delivery_partners AS d JOIN dark_stores AS s ON d.store_id = s.store_id GROUP BY s.city"},
  {"difficulty": "medium", "question": "Which payment method is used most often for orders with an order value above 1000?",
   "query": "SELECT payment_method FROM orders WHERE order_value > 1000 GROUP BY payment_method ORDER BY COUNT(*) DESC LIMIT 1"},
  {"difficulty": "medium", "question": "How many orders were placed between 18:00 and 21:59 (6 PM to before 10 PM)?",
   "query": "SELECT COUNT(*) FROM orders WHERE strftime('%H', order_time) BETWEEN '18' AND '21'"},
  {"difficulty": "medium", "question": "Which brand has sold the highest total quantity of items?",
   "query": "SELECT p.brand FROM order_items AS oi JOIN products AS p ON oi.product_id = p.product_id GROUP BY p.brand ORDER BY SUM(oi.quantity) DESC LIMIT 1"},

  {"difficulty": "hard", "question": "Which dark stores have an average delivery time (over delivered orders) higher than the overall average delivery time of delivered orders? Return the store names.",
   "query": "SELECT s.store_name FROM orders AS o JOIN dark_stores AS s ON o.store_id = s.store_id WHERE o.status = 'delivered' GROUP BY s.store_id HAVING AVG(o.delivery_minutes) > (SELECT AVG(delivery_minutes) FROM orders WHERE status = 'delivered')"},
  {"difficulty": "hard", "question": "How many orders contain both 'FreshFarm Organic Bananas (6 pcs)' and 'FreshFarm Strawberries 200g'?",
   "query": "SELECT COUNT(*) FROM (SELECT oi.order_id FROM order_items AS oi JOIN products AS p ON oi.product_id = p.product_id WHERE p.product_name IN ('FreshFarm Organic Bananas (6 pcs)', 'FreshFarm Strawberries 200g') GROUP BY oi.order_id HAVING COUNT(DISTINCT p.product_name) = 2)"},
  {"difficulty": "hard", "question": "For each pincode that has more than one dark store, what is the average number of orders per store in that pincode?",
   "query": "SELECT s.pincode, COUNT(o.order_id) * 1.0 / COUNT(DISTINCT s.store_id) FROM dark_stores AS s JOIN orders AS o ON o.store_id = s.store_id GROUP BY s.pincode HAVING COUNT(DISTINCT s.store_id) > 1"},
  {"difficulty": "hard", "question": "For each city where a dark store is located, what percentage (0 to 100) of its orders were cancelled?",
   "query": "SELECT s.city, 100.0 * SUM(CASE WHEN o.status = 'cancelled' THEN 1 ELSE 0 END) / COUNT(*) FROM orders AS o JOIN dark_stores AS s ON o.store_id = s.store_id GROUP BY s.city"},
  {"difficulty": "hard", "question": "Which products are below their reorder level (stock quantity less than reorder level) in more than 8 stores? Return the product names.",
   "query": "SELECT p.product_name FROM inventory AS i JOIN products AS p ON i.product_id = p.product_id WHERE i.stock_qty < i.reorder_level GROUP BY p.product_id HAVING COUNT(*) > 8"},
  {"difficulty": "hard", "question": "What is the name of the delivery partner who completed the most delivered orders?",
   "query": "SELECT d.partner_name FROM orders AS o JOIN delivery_partners AS d ON o.partner_id = d.partner_id WHERE o.status = 'delivered' GROUP BY d.partner_id ORDER BY COUNT(*) DESC LIMIT 1"},
  {"difficulty": "hard", "question": "What is the total revenue (quantity times unit price) from perishable products in delivered orders served by dark stores in Hyderabad?",
   "query": "SELECT SUM(oi.quantity * oi.unit_price) FROM order_items AS oi JOIN products AS p ON oi.product_id = p.product_id JOIN orders AS o ON oi.order_id = o.order_id JOIN dark_stores AS s ON o.store_id = s.store_id WHERE p.is_perishable = 1 AND o.status = 'delivered' AND s.city = 'Hyderabad'"},
  {"difficulty": "hard", "question": "What is the name of the customer who spent the most in total order value on delivered orders?",
   "query": "SELECT c.customer_name FROM orders AS o JOIN customers AS c ON o.customer_id = c.customer_id WHERE o.status = 'delivered' GROUP BY c.customer_id ORDER BY SUM(o.order_value) DESC LIMIT 1"},
  {"difficulty": "hard", "question": "In which customer cities is the average order value of Plus members higher than the average order value of non-members?",
   "query": "SELECT c.city FROM orders AS o JOIN customers AS c ON o.customer_id = c.customer_id GROUP BY c.city HAVING AVG(CASE WHEN c.is_plus_member = 1 THEN o.order_value END) > AVG(CASE WHEN c.is_plus_member = 0 THEN o.order_value END)"},
  {"difficulty": "hard", "question": "For each dark store in Pune, how many orders were delivered in under 10 minutes? Show the store name and the count.",
   "query": "SELECT s.store_name, COUNT(*) FROM orders AS o JOIN dark_stores AS s ON o.store_id = s.store_id WHERE s.city = 'Pune' AND o.status = 'delivered' AND o.delivery_minutes < 10 GROUP BY s.store_id"},
  {"difficulty": "hard", "question": "For each product category, how many different customers bought at least one product from it?",
   "query": "SELECT p.category, COUNT(DISTINCT o.customer_id) FROM order_items AS oi JOIN products AS p ON oi.product_id = p.product_id JOIN orders AS o ON oi.order_id = o.order_id GROUP BY p.category"},
  {"difficulty": "hard", "question": "Which dark stores have no delivery partner whose vehicle type is 'bicycle'? Return the store names.",
   "query": "SELECT store_name FROM dark_stores WHERE store_id NOT IN (SELECT store_id FROM delivery_partners WHERE vehicle_type = 'bicycle')"},
  {"difficulty": "hard", "question": "What is the average number of units (sum of item quantities) per delivered order?",
   "query": "SELECT AVG(units) FROM (SELECT SUM(oi.quantity) AS units FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE o.status = 'delivered' GROUP BY oi.order_id)"},
  {"difficulty": "hard", "question": "Which dark store had the highest total order value from delivered orders in June 2026? Give the store name.",
   "query": "SELECT s.store_name FROM orders AS o JOIN dark_stores AS s ON o.store_id = s.store_id WHERE o.status = 'delivered' AND strftime('%Y-%m', o.order_time) = '2026-06' GROUP BY s.store_id ORDER BY SUM(o.order_value) DESC LIMIT 1"},
  {"difficulty": "hard", "question": "What are the top 3 customer cities by number of distinct customers who placed at least one order?",
   "query": "SELECT c.city FROM orders AS o JOIN customers AS c ON o.customer_id = c.customer_id GROUP BY c.city ORDER BY COUNT(DISTINCT c.customer_id) DESC LIMIT 3"}
]

In [ ]:
%%writefile /content/instaquery/templates/README.md.tmpl
# InstaQuery: Fine-tuning a Small Open LLM for Text-to-SQL (QLoRA → vLLM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tarubee19/instaquery/blob/main/notebooks/InstaQuery_Colab.ipynb)
[![tests](https://github.com/tarubee19/instaquery/actions/workflows/ci.yml/badge.svg)](https://github.com/tarubee19/instaquery/actions/workflows/ci.yml)
![Python](https://img.shields.io/badge/python-3.11%20%7C%203.12-blue)
![License: MIT](https://img.shields.io/badge/license-MIT-green)

**Ask a database a question in plain English and get the SQL.** InstaQuery fine-tunes
`{{MODEL}}` with **QLoRA** on the Spider benchmark, measures it with
**execution accuracy** against zero-shot, few-shot and **retrieval-augmented (RAG)** baselines,
quantizes it to **4-bit**, and serves it with **vLLM** behind an OpenAI-compatible API.
Everything runs on **one free Colab T4 GPU**.

{{HEADLINE}}

## Results

All numbers below were produced by `notebooks/InstaQuery_Colab.ipynb` (run mode: `{{RUN_MODE}}`, GPU: {{GPU}}).

{{RESULTS_TABLE}}

![Execution accuracy by variant](assets/ex_by_variant.png)

![Accuracy by Spider difficulty](assets/ex_by_difficulty.png)

| Training | |
|---|---|
{{TRAIN_TABLE}}

![Training loss](assets/training_loss.png)

### Serving (vLLM vs Hugging Face generate, same GPU)

{{BENCH_TABLE}}

![Throughput](assets/throughput.png)

{{ERROR_ANALYSIS}}

{{FINDINGS}}

## How it works

```mermaid
flowchart LR
  A[Spider train<br/>8.6K question/SQL pairs] --> B[Prompt builder<br/>schema + sample values + question]
  B --> C[QLoRA fine-tuning<br/>4-bit NF4 base + LoRA r=16]
  C --> D[Merge LoRA into fp16]
  D --> E[vLLM server<br/>OpenAI-compatible API]
  D --> F[4-bit NF4 quantized model]
  E --> G[Execution-accuracy evaluator<br/>Spider dev + quick-commerce DB]
  F --> G
  H[Baselines: zero-shot,<br/>few-shot, RAG few-shot] --> G
```

1. **Data.** [Spider 1.0](https://yale-lily.github.io/spider): 8,659 training questions over 146 databases and
   1,034 dev questions over 20 *different* databases, so the dev score measures generalisation to unseen schemas.
   A second test set is a synthetic **quick-commerce** SQLite database (dark stores, orders, inventory,
   delivery partners; 7 tables, 25K orders) with 50 hand-written, execution-verified business questions.
2. **Prompt.** The schema is serialised as `CREATE TABLE` statements with primary/foreign keys and three sample
   values per text column (so the model can match literals). Very large schemas drop the sample values.
3. **Baselines.** Same base model with (a) zero-shot prompting, (b) three fixed examples, and
   (c) **RAG**: the three most similar training questions retrieved with dense embeddings
   (`BAAI/bge-small-en-v1.5`), never from the same database.
4. **QLoRA.** Base weights frozen in 4-bit NF4 with double quantisation; LoRA adapters (r=16, α=32, dropout 0.05)
   on all attention and MLP projections; loss only on SQL tokens; cosine LR 2e-4, effective batch 16, 1 epoch,
   paged 8-bit AdamW, gradient checkpointing.
5. **Evaluation.** A prediction is correct if it returns the same result as the gold query when executed
   (bag semantics, order-sensitive only with `ORDER BY`, column permutations allowed), following the official
   Spider test-suite evaluator. Queries run read-only with a hard timeout. Difficulty labels and exact-set-match
   come from the official evaluator.
6. **Serving.** The LoRA weights are merged into an fp16 model, served with vLLM (continuous batching,
   PagedAttention), and benchmarked at different concurrency levels against Hugging Face `generate`.
7. **Quantisation.** The merged model is reloaded in 4-bit NF4 to measure the accuracy/memory trade-off.
8. **Demo.** A Gradio app turns questions into SQL for the quick-commerce database, with guardrails
   (sqlglot parse, single read-only `SELECT`, read-only connection, timeout, row cap).

{{ENGINEERING}}

## Reproduce

Click the **Open in Colab** badge (or open `notebooks/InstaQuery_Colab.ipynb`), choose **Runtime → Change runtime type → T4 GPU**,
set `RUN_MODE = "quick"` for a 20-minute smoke test or `"full"` for the complete run, then **Run all**.
Outputs (metrics, predictions, charts, LoRA adapter) are saved to Google Drive.

```bash
pip install -r requirements.txt
pytest -q            # unit tests for evaluator, prompts, schema, guardrails, domain DB
python app/gradio_app.py --model <merged_model_dir> --db quickcommerce.sqlite
```

## Repository layout

```
src/instaquery/
  spider_data.py   download + load Spider
  schema.py        SQLite schema -> prompt text (keys, sample values)
  prompts.py       chat prompts, SQL extraction/normalisation
  retrieval.py     dense / TF-IDF retrieval of similar examples (RAG few-shot)
  train.py         QLoRA fine-tuning + LoRA merge
  generation.py    batched HF generation and async OpenAI-compatible client
  serving.py       vLLM server in an isolated venv
  sql_eval.py      execution accuracy with timeouts, error taxonomy
  official_eval.py official Spider difficulty + exact-match
  domain_db.py     synthetic quick-commerce database
  safety.py        SQL guardrails for the demo
  pipeline.py      cached evaluation runs and benchmarks
  report.py        tables, charts, README, CV summary
data/quickcommerce_eval.json   50 business questions with verified gold SQL
notebooks/InstaQuery_Colab.ipynb  end-to-end Colab notebook (T4 GPU)
results/                       metrics, benchmark, training stats from the full run
assets/                        charts used in this README
app/gradio_app.py              demo UI
tests/                         pytest suite (runs in CI)
```

## Limitations

* Spider dev is a public benchmark; scores are not comparable to the hidden Spider test leaderboard.
* The quick-commerce set has 50 questions, so its numbers have wide confidence intervals (±7 pts at 95%).
* The quick-commerce database is synthetic (fictional brands and people).
* Exact-set-match is reported for reference only; execution accuracy is the main metric.

## References

Yu et al., *Spider* (EMNLP 2018) · Zhong et al., *Semantic Evaluation for Text-to-SQL with Distilled Test Suites* (EMNLP 2020) ·
Hu et al., *LoRA* (2021) · Dettmers et al., *QLoRA* (2023) · Kwon et al., *vLLM / PagedAttention* (SOSP 2023) ·
Hui et al., *Qwen2.5-Coder Technical Report* (2024).

In [ ]:
%%writefile /content/instaquery/app/gradio_app.py
"""InstaQuery demo: ask a question in English, get SQL + results from the quick-commerce database.

Run locally:   python app/gradio_app.py --model <merged_model_dir_or_hf_id> --db quickcommerce.sqlite
Or point it at a running vLLM server:  --api http://127.0.0.1:8000/v1
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from instaquery.prompts import build_messages, extract_sql  # noqa: E402
from instaquery.safety import check_sql  # noqa: E402
from instaquery.schema import serialize_schema  # noqa: E402
from instaquery.sql_eval import execute_sql  # noqa: E402

EXAMPLES = [
    "Which 5 products generated the most revenue?",
    "What is the average delivery time of delivered orders in each city?",
    "Which pincodes have more than one dark store, and how many orders did each get?",
    "How many orders contain both bananas and strawberries?",
    "Which delivery partners in Pune have a rating above 4.8?",
]


def build_demo(generate_fn, db_path: str):
    import gradio as gr
    import pandas as pd

    schema = serialize_schema(str(db_path), 3)

    def answer(question: str):
        if not question.strip():
            return "", None, "Type a question."
        t0 = time.perf_counter()
        raw = generate_fn(build_messages(schema, question))
        sql = extract_sql(raw)
        ok, why = check_sql(sql)
        if not ok:
            return sql, None, f"Blocked by guardrail: {why}"
        flag, res = execute_sql(db_path, sql, timeout_s=10)
        dt = time.perf_counter() - t0
        if flag != "ok":
            return sql, None, f"SQL error: {res}"
        df = pd.DataFrame(res[:200])
        return sql, df, f"{len(res)} rows in {dt:.2f}s (read-only database, 10s timeout)"

    with gr.Blocks(title="InstaQuery: Text-to-SQL") as demo:
        gr.Markdown("## InstaQuery: ask the quick-commerce database in plain English\n"
                    "A QLoRA fine-tuned small LLM writes the SQL; queries run read-only with guardrails.")
        q = gr.Textbox(label="Question", placeholder=EXAMPLES[0])
        btn = gr.Button("Generate SQL and run", variant="primary")
        sql_out = gr.Code(label="Generated SQL", language="sql")
        table = gr.Dataframe(label="Result (first 200 rows)")
        status = gr.Markdown()
        gr.Examples(EXAMPLES, inputs=q)
        btn.click(answer, q, [sql_out, table, status])
        q.submit(answer, q, [sql_out, table, status])
    return demo


def _hf_generate_fn(model_dir: str):
    from instaquery.generation import HFGenerator, load_hf_model
    model, tok = load_hf_model(model_dir)
    gen = HFGenerator(model, tok, batch_size=1, max_new_tokens=256)
    return lambda msgs: gen.generate_one_timed(msgs).text


def _api_generate_fn(base_url: str, model: str):
    from instaquery.generation import OpenAIGenerator
    gen = OpenAIGenerator(base_url, model)
    return lambda msgs: gen.generate([msgs])[0].text


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--db", required=True)
    ap.add_argument("--model", default=None)
    ap.add_argument("--api", default=None)
    ap.add_argument("--served-name", default="instaquery")
    a = ap.parse_args()
    fn = _api_generate_fn(a.api, a.served_name) if a.api else _hf_generate_fn(a.model)
    build_demo(fn, a.db).launch()

In [ ]:
%%writefile /content/instaquery/tests/test_core.py
import json
import sqlite3
import sys
from pathlib import Path

import pytest

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

from instaquery.domain_db import build_quickcommerce_db  # noqa: E402
from instaquery.prompts import build_messages, extract_sql, normalize_sql  # noqa: E402
from instaquery.retrieval import ExampleRetriever, pick_fixed_shots  # noqa: E402
from instaquery.safety import check_sql  # noqa: E402
from instaquery.schema import serialize_schema  # noqa: E402
from instaquery.sql_eval import categorize_error, exec_match, execute_sql, result_eq, score  # noqa: E402


@pytest.fixture(scope="module")
def db(tmp_path_factory):
    return build_quickcommerce_db(tmp_path_factory.mktemp("qc") / "qc.sqlite", n_customers=400, n_orders=3000)


# ------------------------------------------------------------ prompts
def test_extract_sql_variants():
    assert extract_sql("```sql\nSELECT a FROM t;\n```") == "SELECT a FROM t"
    assert extract_sql("SQL: SELECT  a\n FROM t ; -- done") == "SELECT a FROM t"
    assert extract_sql("Here is the query:\nSELECT x FROM y WHERE z = 'a;b'") == "SELECT x FROM y WHERE z = 'a;b'"
    assert extract_sql("WITH c AS (SELECT 1) SELECT * FROM c") == "WITH c AS (SELECT 1) SELECT * FROM c"
    assert extract_sql("SELECT a FROM t\n\nThis query selects a.") == "SELECT a FROM t"


def test_normalize_keeps_string_literals():
    assert normalize_sql("SELECT  a FROM t WHERE n = 'two  spaces' ;") == "SELECT a FROM t WHERE n = 'two  spaces'"


def test_messages_shape():
    m = build_messages("CREATE TABLE t (a INT);", "How many?", [{"question": "q", "query": "SELECT 1"}], "SELECT count(*) FROM t")
    assert [x["role"] for x in m] == ["system", "user", "assistant"]
    assert "Q: q" in m[1]["content"] and m[2]["content"] == "SELECT count(*) FROM t"


# ------------------------------------------------------------ schema
def test_schema_has_tables_values_and_fks(db):
    s = serialize_schema(str(db), 3)
    for t in ["dark_stores", "products", "orders", "order_items", "inventory"]:
        assert f"CREATE TABLE {t} (" in s
    assert "-- e.g." in s and "FOREIGN KEY (store_id) REFERENCES dark_stores(store_id)" in s
    assert "-- e.g." not in serialize_schema(str(db), 0)


# ------------------------------------------------------------ evaluation
def test_result_eq_semantics():
    assert result_eq([(1, "a"), (2, "b")], [(2, "b"), (1, "a")], order_matters=False)
    assert not result_eq([(1, "a"), (2, "b")], [(2, "b"), (1, "a")], order_matters=True)
    assert result_eq([(1, "a")], [("a", 1)], order_matters=False)          # column permutation
    assert not result_eq([(1,), (1,)], [(1,)], order_matters=False)        # bag semantics
    assert result_eq([], [], order_matters=False)


def test_exec_match_and_errors(db):
    gold = "SELECT COUNT(*) FROM dark_stores WHERE city = 'Bengaluru'"
    assert exec_match(db, "SELECT count(store_id) FROM dark_stores WHERE city='Bengaluru'", gold).correct
    r = exec_match(db, "SELECT COUNT(*) FROM stores", gold)
    assert not r.correct and categorize_error(r) == "unknown table"
    r = exec_match(db, "SELECT COUNT(*) FROM dark_stores WHERE city = 'Mumbai'", gold)
    assert r.pred_executable and categorize_error(r) == "wrong result (values)"


def test_timeout_stops_runaway_query(db):
    flag, msg = execute_sql(db, "SELECT COUNT(*) FROM order_items a, order_items b, order_items c", timeout_s=0.5)
    assert flag == "error" and msg == "timeout"


def test_readonly(db):
    flag, msg = execute_sql(db, "DELETE FROM orders")
    assert flag == "error"
    assert sqlite3.connect(db).execute("SELECT COUNT(*) FROM orders").fetchone()[0] == 3000


def test_domain_gold_queries_all_execute(db):
    qs = json.loads((ROOT / "data" / "quickcommerce_eval.json").read_text())
    assert len(qs) == 50
    for q in qs:
        flag, res = execute_sql(db, q["query"])
        assert flag == "ok", (q["question"], res)


def test_score_gold_is_perfect(db):
    qs = json.loads((ROOT / "data" / "quickcommerce_eval.json").read_text())
    exs = [dict(q, db_id="qc") for q in qs]
    sc = score(exs, [q["query"] for q in qs], lambda _: db, progress=False, keep_distinct=True)
    assert sc.ex == 1.0 and sc.exec_rate == 1.0


# ------------------------------------------------------------ safety
@pytest.mark.parametrize("sql,ok", [
    ("SELECT * FROM orders", True),
    ("WITH x AS (SELECT 1) SELECT * FROM x", True),
    ("DROP TABLE orders", False),
    ("DELETE FROM orders WHERE 1=1", False),
    ("SELECT 1; DROP TABLE orders", False),
    ("UPDATE orders SET status='x'", False),
    ("", False),
])
def test_guardrail(sql, ok):
    assert check_sql(sql)[0] is ok


# ------------------------------------------------------------ retrieval
def test_tfidf_retriever_excludes_same_db():
    pool = [{"question": "how many singers are there", "query": "SELECT count(*) FROM singer", "db_id": "a"},
            {"question": "how many singers do we have", "query": "SELECT count(*) FROM singer s", "db_id": "b"},
            {"question": "list all cars", "query": "SELECT * FROM cars", "db_id": "c"}]
    r = ExampleRetriever(pool, model_name=None)
    got = r.retrieve_many(["how many singers"], k=1, exclude_db=["a"])[0]
    assert got[0]["db_id"] == "b"


def test_pick_fixed_shots_deterministic():
    pool = [{"question": f"q{i}", "query": q, "db_id": "x"} for i, q in enumerate([
        "SELECT name FROM t", "SELECT count(*) FROM t WHERE a > 1",
        "SELECT a FROM t GROUP BY a HAVING count(*) > 2",
        "SELECT T1.a FROM t AS T1 JOIN s AS T2 ON T1.id = T2.id ORDER BY T1.b DESC LIMIT 1"])]
    shots = pick_fixed_shots(pool)
    assert [s["question"] for s in shots] == ["q1", "q2", "q3"]

In [ ]:
!cd /content/instaquery && python -m pytest -q tests 2>&1 | tail -3

## 3. Configuration and data

In [ ]:
import json, sys, importlib
from pathlib import Path
sys.path.insert(0, "/content/instaquery/src")
import instaquery
from instaquery.config import load_config
from instaquery.pipeline import Workspace
cfg = load_config(run_mode=RUN_MODE, output_dir=OUTPUT_DIR, base_model=BASE_MODEL)
cfg.save()
ws = Workspace(cfg, Path("/content/instaquery/data/quickcommerce_eval.json")).prepare_data()

In [ ]:
from instaquery.official_eval import ensure_repo, add_hardness
OFFICIAL = ensure_repo(Path(cfg.work_dir) / "test-suite-sql-eval")
add_hardness(ws.dev, OFFICIAL, ws.db_lookup)
from collections import Counter
print("Spider dev difficulty:", dict(Counter(e["hardness"] for e in ws.dev)))
print("Quick-commerce difficulty:", dict(Counter(e["hardness"] for e in ws.domain)))

In [ ]:
# Dataset statistics + one full prompt
import numpy as np, matplotlib.pyplot as plt
tok = ws.load_tokenizer()
sample = ws.dev[:300] + ws.train[:300]
lens = [len(tok(tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True))["input_ids"])
        for m in ws.messages_for(sample, "zero")]
stats = {"train_questions": len(ws.train), "train_databases": len({e["db_id"] for e in ws.train}),
         "dev_questions": len(ws.dev), "dev_databases": len({e["db_id"] for e in ws.dev}),
         "quickcommerce_questions": len(ws.domain),
         "prompt_tokens_mean": float(np.mean(lens)), "prompt_tokens_p95": float(np.percentile(lens, 95))}
cfg.out("dataset_stats.json").write_text(json.dumps(stats, indent=2)); print(stats)
fig, ax = plt.subplots(figsize=(7, 3.2), facecolor="#fcfcfb")
ax.hist(lens, bins=40, color="#2a78d6", edgecolor="#fcfcfb", linewidth=1)
ax.set_xlabel("Prompt length (tokens)"); ax.set_ylabel("Questions"); ax.set_title("Prompt length distribution", loc="left")
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(cfg.out("assets", "prompt_lengths.png"), dpi=150); plt.show()
print(tok.apply_chat_template(ws.messages_for(ws.dev[:1], "rag")[0], tokenize=False, add_generation_prompt=True))

## 4. vLLM (isolated environment)
vLLM is installed into its own virtual env so it cannot clash with the training stack. If it fails, everything falls back to Hugging Face generation automatically.

In [ ]:
from instaquery.serving import install_vllm
VLLM_OK = install_vllm(cfg.vllm_version, log_file=cfg.out("logs", "vllm_install.log"))
ws.backend = "vllm" if VLLM_OK else "hf"
print("generation backend:", ws.backend)

## 5. Baselines with the base model
Zero-shot, 3 fixed examples, and RAG (3 most similar training questions, never from the same database).

In [ ]:
base_plan = [("base_zeroshot", "spider_dev", "zero"), ("base_fewshot", "spider_dev", "fixed"),
             ("base_rag", "spider_dev", "rag"), ("base_zeroshot", "quickcommerce", "zero"),
             ("base_rag", "quickcommerce", "rag")]
base_results = ws.run_plan(cfg.base_model, base_plan, tag="base")
for m in base_results:
    print(f"{m['dataset']:14s} {m['variant']:15s} EX={100*m['execution_accuracy']:.1f}%")

## 6. QLoRA fine-tuning

In [ ]:
from instaquery.train import train_qlora, merge_adapter
stats_file = cfg.adapter_dir / "train_stats.json"
if stats_file.exists():
    train_stats = json.loads(stats_file.read_text()); print("[train] cached adapter found")
else:
    train_stats = train_qlora(cfg, ws.train, ws.schema_for, cfg.adapter_dir, Path(cfg.output_dir) / "checkpoints")
print(json.dumps(train_stats, indent=2))

In [ ]:
from instaquery import report
report.chart_training_loss(json.loads((cfg.adapter_dir / "log_history.json").read_text()), cfg.out("assets", "training_loss.png"))
from IPython.display import Image, display
display(Image(str(cfg.out("assets", "training_loss.png"))))

In [ ]:
merged = merge_adapter(cfg.base_model, cfg.adapter_dir, cfg.merged_dir)

## 7. Evaluate the fine-tuned model + vLLM serving benchmark

In [ ]:
from instaquery.generation import OpenAIGenerator
bench_file = Path(cfg.output_dir) / "benchmark.json"
bench = json.loads(bench_file.read_text()) if bench_file.exists() else {}
def _bench(gen):
    if isinstance(gen, OpenAIGenerator) and "vllm" not in bench:
        bench["vllm"] = ws.bench_vllm(gen)
ft_plan = [("finetuned", "spider_dev", "zero"), ("finetuned_rag", "spider_dev", "rag"),
           ("finetuned", "quickcommerce", "zero")]
ft_results = ws.run_plan(str(merged), ft_plan, tag="finetuned", after=None if "vllm" in bench else _bench)
if "hf_batched" not in bench:
    bench.update(ws.bench_hf(str(merged)))
bench_file.write_text(json.dumps(bench, indent=2))
for m in ft_results:
    print(f"{m['dataset']:14s} {m['variant']:15s} EX={100*m['execution_accuracy']:.1f}%")
print(json.dumps(bench, indent=2))

## 8. 4-bit NF4 quantization: accuracy vs memory

In [ ]:
nf4_results = ws.run_plan(str(merged), [("finetuned_nf4", "spider_dev", "zero"), ("finetuned_nf4", "quickcommerce", "zero")],
                          tag="nf4", backend="hf", quant_4bit=True)
for m in nf4_results:
    print(f"{m['dataset']:14s} {m['variant']:15s} EX={100*m['execution_accuracy']:.1f}%  memory={m.get('model_memory_gb', 0):.2f} GB")

## 9. Optional: paid API reference
Add an `OPENAI_API_KEY` in Colab **Secrets** (key icon on the left) to compare with a hosted model. Skipped otherwise.

In [ ]:
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception as e:
    print("No API key configured - skipping API reference.")
if api_key:
    api = OpenAIGenerator(cfg.api_base_url, cfg.api_model, api_key=api_key, concurrency=16)
    for d in ("spider_dev", "quickcommerce"):
        try:
            ws.run_variant(api, "api_zeroshot", d, "zero", extra_meta={"api_model": cfg.api_model})
        except Exception as e:
            print("API evaluation failed:", e)

## 10. Official Spider exact-set-match (reference metric)

In [ ]:
em_file = Path(cfg.output_dir) / "official_exact_match.json"
official_em = json.loads(em_file.read_text()) if em_file.exists() else {}
from instaquery.official_eval import exact_match
for v in ("base_zeroshot", "finetuned"):
    p = Path(cfg.output_dir) / "predictions" / "spider_dev" / f"{v}.jsonl"
    if v in official_em or not p.exists():
        continue
    preds = [json.loads(l)["pred"] for l in p.read_text().splitlines()]
    res = exact_match(ws.dev, preds, OFFICIAL, Path(ws.spider_root) / "database", Path(ws.spider_root) / "tables.json",
                      Path(cfg.work_dir) / f"em_{v}")
    if res: official_em[v] = res
em_file.write_text(json.dumps(official_em, indent=2)); print(official_em)

## 11. Report: tables, charts, README, CV bullets

In [ ]:
from IPython.display import Markdown, Image, display
importlib.reload(report)
metrics = report.load_metrics(cfg.output_dir)
assets = Path(cfg.output_dir) / "assets"
report.chart_ex_by_variant(metrics, assets / "ex_by_variant.png")
report.chart_by_difficulty(metrics, assets / "ex_by_difficulty.png")
report.chart_throughput(bench, assets / "throughput.png")
err_md = report.error_analysis(cfg.output_dir)
readme = report.build_readme(Path("/content/instaquery/templates/README.md.tmpl").read_text(), cfg, metrics,
                             train_stats, bench, err_md, official_em)
cv = report.cv_bullets(metrics, train_stats, bench, cfg.base_model.split("/")[-1])
(Path(cfg.output_dir) / "README.md").write_text(readme)
(Path(cfg.output_dir) / "cv_bullets.md").write_text(cv)
(Path(cfg.output_dir) / "summary_table.md").write_text(report.summary_table(metrics))
display(Markdown(report.summary_table(metrics)))
for f in ("ex_by_variant.png", "ex_by_difficulty.png", "throughput.png"):
    if (assets / f).exists(): display(Image(str(assets / f)))
display(Markdown(err_md))
display(Markdown("### CV bullets (real numbers from this run)\n" + cv))

## 12. Demo: ask the quick-commerce database in English

In [ ]:
%pip install -q gradio
import sys
sys.path.insert(0, "/content/instaquery/app")
from gradio_app import build_demo
from instaquery.generation import HFGenerator, load_hf_model
demo_model, demo_tok = load_hf_model(str(merged))
demo_gen = HFGenerator(demo_model, demo_tok, batch_size=1, max_new_tokens=256)
demo = build_demo(lambda msgs: demo_gen.generate_one_timed(msgs).text, str(cfg.domain_db_path))
demo.launch(share=False, inline=True, height=750)

## 13. Package results

In [ ]:
import shutil
ws.timings["total_runtime_min"] = (time.time() - T_START) / 60
(Path(cfg.output_dir) / "timings.json").write_text(json.dumps(ws.timings, indent=2))
pkg = Path("/content/instaquery_results")
if pkg.exists(): shutil.rmtree(pkg)
shutil.copytree(cfg.output_dir, pkg, ignore=shutil.ignore_patterns("checkpoints", "*.safetensors", "*.bin", "optimizer.pt"))
shutil.copytree("/content/instaquery", pkg / "repo", ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache"))
shutil.copy(cfg.domain_db_path, pkg / "quickcommerce.sqlite")
zip_path = shutil.make_archive("/content/instaquery_results", "zip", pkg)
if USE_DRIVE: shutil.copy(zip_path, DRIVE_DIR)
print("Results zip:", zip_path, "| total runtime %.0f min" % ws.timings["total_runtime_min"])
print("ALL DONE")